# BetaTour run v27 (pilot): birleşim grafında tam çözüm

Run v26'da altı LKH-3 sürecinin turlarını bütçenin yarısında birleştirmek, süreçleri bütçe sonuna kadar çalıştırmaktan
daha kısa turlar verdi; GA-EAX ile LKH-3'ün kendi tur birleştirmesi arasındaki fark anlamlı çıkmadı. Bu pilot iki soruyu
sorar:

1. **Matematik:** altı sürecin en iyi turlarının **birleşim grafındaki en kısa tur** (SCIP ile dal-ve-kesi, kanıtlanmış
   en iyi) GA-EAX'in bulduğu turdan daha kısa mı?
2. **Derin öğrenme:** birleşim grafına BetaTour'un en olası yeni kenarlarını (n'nin %2'si) eklemek, aynı sayıda en kısa
   yeni kenar eklemekten ve hiç eklememekten daha kısa tur veriyor mu?

Kollar (her örnekte tek bir kayıtlı koşudan; kesme anı bütçenin yarısı): `lkh6` (altı süreç bütçe sonuna kadar),
`trials_50` ve `best6_50` (GA-EAX; 102 tur ve altı en iyi tur), `dual_50` (ikisinden kısa olanı; run v26'nın birincil
kolu), `lkhmerge_50` (LKH-3'ün kendi tur birleştirmesi), `exact6_50` (birleşimde tam çözüm), `exact6beta_50`
(birleşim + BetaTour kenarları), `exact6near_50` (birleşim + en kısa kenarlar). Örnekler: 12 yeni 10.000 düğümlük
örnek (6 düzgün, 6 kümeli; 120 s) ve 2 yeni 20.000 düğümlük örnek (240 s; betimsel).

Okumalar ve karar kuralı `experiments/PROTOCOL_v27.md` dosyasındadır; defter, protokol ve defterin kendisi GitHub'da
(main) kurulduğu biçimde değilse başlamaz. Önce depoyu commit ve push edin.

**Çalıştırma:** Çalışma zamanı türü **L4 GPU + Yüksek RAM** (12 vCPU, 53 GB) seçin ve bütün hücreleri sırayla
çalıştırın. Tahmini süre: kurulum (PySCIPOpt kurulumu, LKH-3 ve GA-EAX derlemeleri, model) 6-8 dk; öz sınama 2-3 dk;
10.000 düğümlük 12 örnek, her biri yaklaşık 5 dk, 1 saat; 20.000 düğümlük 2 örnek, her biri yaklaşık 10 dk, 20 dk;
çözümleme ve arşiv 1 dk; toplam yaklaşık 1,5 saat. Biten örnekler Drive'a yazılır
(`MyDrive/BetaTour/v27_union14`) ve defter yeniden çalıştırılınca atlanır; makine değişirse yarım kalan örnek baştan
koşar. İlk hücredeki `MAX_NEW_INSTANCES` deneyi oturumlara böler. Sonunda indirilen
`v27_union14_results.zip` dosyasını gönderin.

In [ ]:
# Colab çalışma ortamı: L4 GPU + Yüksek RAM (12 vCPU, 53 GB; run v26 ile aynı ortam). GPU yalnızca BetaTour çıkarımı
# için kullanılır (örnek başına bir kez, 1 s'den az). Deney bitene kadar aynı ortam türünü kullanın.
EXPERIMENT_NAME = "v27_union14"
MAX_NEW_INSTANCES = None      # örn. 7: deneyi kısa oturumlara böler; biten örnekler Drive'dan okunur ve atlanır
SAVE_TO_DRIVE = True
ALLOW_HARDWARE_CHANGE = False  # True: sonraki oturum başka makine türünde sürebilir (rapora not düşülür)
REQUIRE_PUSHED_PROTOCOL = True  # kurulum, protokol GitHub'da (main) bu defterin protokolüyle aynı değilse durur

W = 6                         # paralel LKH süreci (tohum) sayısı
SEEDS = list(range(201, 207))        # kayıt eden altı LKH süreci (run v26 ile aynı)
MERGE_SEEDS = list(range(301, 307))  # LKH'nin kendi tur birleştirmesini yapan altı süreç (run v26 ile aynı)
# (grup, düğüm sayısı, örnek sayısı, aile, üretici tohumu, bütçe s). Tohumların hiçbiri önceki koşularda ya da ön
# sınamada kullanılmadı. Bütçe: düğüm başına 12 ms (run v26 gibi).
SYNTHETIC = [("uni10k", 10000, 6, "uniform", 11111, 120.0), ("clu10k", 10000, 6, "clustered", 11212, 120.0),
             ("uni20k", 20000, 1, "uniform", 11313, 240.0), ("clu20k", 20000, 1, "clustered", 11414, 240.0)]
DECISION_GROUPS = ["uni10k", "clu10k"]   # okumalar (karar kuralı) bu 12 örnek üzerinde; 20.000 düğüm betimseldir
K_LIST = 5
K_GRAPH = 16                  # BetaTour grafiği: 16 en yakın komşu (+ modelin topluluk kenarları), run v21 gibi
SPLIT = 0.5                   # LKH süreçlerinin durdurulduğu an (bütçenin kesri); run v26 gibi
PER_PROCESS = 17              # GA-EAX "trials" popülasyonu: süreç başına en iyi tur + son 16 deneme turu (run v26)
EXTRA_FRACTION = 0.02         # birleşim grafına eklenen kenar sayısı: düğüm sayısının %2'si (model ya da en yakın)
EAX_KIDS = 30                 # GA-EAX: ebeveyn çifti başına çocuk sayısı (yazarın önerdiği değer)
EAX_NORMAL_MAX_N = 20000      # bu boyuta kadar GA-EAX'in "Normal" sürümü, üstünde "Large" sürümü (bellek)
EAX_SCRATCH_POP = 300         # yalnızca öz sınamada (GA-EAX'in duvar saati sınırı denetimi) kullanılır
STOP_PAD_S = 2.0              # LKH süreçleri sürelerinden bu kadar sonra dışarıdan durdurulur
EXACT_PAD_S = 60.0            # tam çözüm süreçleri kendi sınırlarından bu kadar sonra dışarıdan durdurulur (güvenlik)
REQUIRE_12_VCPUS = True
MIN_RAM_GB = 24.0
PYSCIPOPT_VERSION = "6.3.0"   # SCIP 10.0 (pip tekerleğiyle gelir)
REPO_COMMIT = "d63f7e17942a6af5cba14ad010edca1069d477b3"
CHECKPOINT_SHA256 = "2322f66c752913f6179f8c61f6b50b0f9d21426857d3ec0155e02891fb3ebd70"
PROTOCOL_PATH = "experiments/PROTOCOL_v27.md"
PROTOCOL_SHA256 = "3bf5459b2675ba4e50289b33bed342ae938d065ec9099e056a09e87b8c04f538"    # depodaki protokol dosyasının özeti (defter kurulurken yazılır)
CODE_SHA256_EXPECTED = "f4cd5d0534ae7f6ba74438c2ddf870117ed65c7e27aa65840b39b7c519e21c30"   # sonraki hücrenin yazdığı byprod_v26.py dosyasının özeti (run v26'nın modülü)
UNION_SHA256_EXPECTED = "3b505805d2c69aa4ea9a1ce00b1192534f1c7a3b893afd7e8ed7f3010d0e3ee6"  # ondan sonraki hücrenin yazdığı unionx_v27.py dosyasının özeti
NOTEBOOK_BUILD = "bf3d4706fa7668c71108e8b8953768125610e64ab377da15c9dbb86e890fad32"      # bu defterin bütün kod hücrelerinin özeti

import os

TEST_MODE = os.environ.get("BETATOUR_V27_TEST") == "1"    # yalnızca yerel öz sınama; Colab'da kapalıdır
if TEST_MODE:
    EXPERIMENT_NAME = "v27_selftest"
    SAVE_TO_DRIVE = False
    REQUIRE_12_VCPUS = False
    REQUIRE_PUSHED_PROTOCOL = False
    MIN_RAM_GB = 0.0
    W = int(os.environ.get("BETATOUR_V27_W", "2"))
    SEEDS, MERGE_SEEDS = SEEDS[:W], MERGE_SEEDS[:W]
    SYNTHETIC = [("uni10k", int(os.environ.get("BETATOUR_V27_N", "1200")), int(os.environ.get("BETATOUR_V27_COUNT", "2")),
                  "uniform", 11111, float(os.environ.get("BETATOUR_V27_BUDGET", "16"))),
                 ("clu20k", 2500, 1, "clustered", 11414, 24.0)]
    DECISION_GROUPS = ["uni10k"]
    PER_PROCESS = int(os.environ.get("BETATOUR_V27_PER_PROCESS", "5"))
    EAX_NORMAL_MAX_N = 2000                               # 2.500 düğümlü örnek "Large" sürümünü sınar
    EAX_SCRATCH_POP = 30
    EXACT_PAD_S = 20.0
assert W == len(SEEDS) == len(MERGE_SEEDS) and 0.0 < SPLIT < 1.0 and PER_PROCESS >= 2 and 0.0 < EXTRA_FRACTION < 1.0

In [ ]:
%%writefile byprod_v26.py
"""Helpers of run v26 (confirmation run): the tours of LKH-3 processes, recorded and recombined with GA-EAX.

Every trial of LKH-3 ends in a locally optimal tour. LKH-3 merges it with its best tour and keeps only the best
tour. This module is the module of run v25 (pilot) with one addition to the recording patch (a mode that records
the new best tours only, used for the processes of the LKH-3 tour-merging control) and with helpers for that
control and for the Holm correction. It

  * patches FindTour.c of LKH-3.0.14 so that a copy of LKH-3 writes the tour of every trial and every new best tour
    to a file (nothing of the search state is changed),
  * patches env.cpp of GA-EAX (Nagata and Kobayashi, 2013) so that it prints a wall-clock time stamp per generation
    and stops at a wall-clock deadline with its best tour,
  * patches evaluator.cpp of GA-EAX so that the lists of nearest cities are found with a grid instead of a scan of
    all cities for every list entry (the same lists; at 50,000 nodes the scan takes about a minute),
  * reads the recorded tours, builds populations from them and runs GA-EAX on a population,
  * holds the paired statistics used by the analysis cell.

The patches are applied by exact string replacement after the SHA-256 of the original file has been compared with
the value below, so another source version is refused and never patched wrongly.

Pure Python and NumPy; tsplab is not imported here.
"""

from __future__ import annotations

import hashlib
import os
import re
import subprocess
import time
from math import comb

import numpy as np

VERSION = "v26-byprod-1"

# ----------------------------------------------------------------------------------------------- source patches
LKH_FINDTOUR_SHA256 = "567e72acc9b7392488dd158be4df0267560424204a0e88d2d9b62f923a7c0eee"      # LKH-3.0.14
EAX_COMMIT = "015dfbe9f267230f78787bd244af393ffc018900"                                     # GA-EAX, 2021-12-29
EAX_ENV_SHA256 = {"Normal": "f923055933b26b3bde02bd1467abca92e99a357d590e62f8044f047ffc8fa6e5",
                  "Large": "5c58951f29477c34f0bd10364147e70ce475899799324775c7df98ddabeae35c"}
EAX_EVALUATOR_SHA256 = {"Normal": "85c07c041c7ea32d42d3b180d1eed30d25707bfeb01393c010803ffbcdcd540c",
                        "Large": "dd1582b36c0c595209cf6d67c287f20000dc24737cd252b474bb03e1763b9767"}
EAX_SOURCES = ["main.cpp", "env.cpp", "cross.cpp", "evaluator.cpp", "indi.cpp", "rand.cpp", "kopt.cpp", "sort.cpp"]
# SHA-256 over the names and contents of all .cpp and .h files of a version (see source_digest).
EAX_TREE_SHA256 = {"Normal": "d4d016b831745a54bd39c458c447e55ac4039e8606220d6fbb2b4ecd130b5813",
                   "Large": "bc5785bdb728b8d46472480eaa5e962faa547b99b02b4d47055ac58971af92af"}

_LKH_FUNCTION = r'''
#include <sys/time.h>

/* Runs v25 and v26: record tours without changing the search.
 * Kind 0: the locally optimal tour of the trial that has just ended (before it is merged with the best tour).
 * Kind 1: the new best tour of the run.
 * Record: five 64-bit integers (kind, trial, cost, milliseconds of LKH's clock since the start of the search,
 * milliseconds of the wall clock since the epoch), then Dimension 32-bit node numbers in tour order.
 * With the environment variable BETATOUR_BEST_ONLY set, records of kind 0 are not written (run v26). */
static void BetaTourRecord(int Kind, GainType RecordCost, double RecordEntryTime)
{
    static FILE *File = 0;
    static int Tried = 0, BestOnly = 0;
    static int *Buffer = 0;
    long long Head[5];
    struct timeval Now;
    int i;

    if (!Tried) {
        char *Name = getenv("BETATOUR_TRIAL_DUMP");
        Tried = 1;
        BestOnly = getenv("BETATOUR_BEST_ONLY") != 0;
        if (Name)
            File = fopen(Name, "wb");
        if (File)
            Buffer = (int *) malloc((size_t) Dimension * sizeof(int));
    }
    if (!File || !Buffer || (Kind == 0 && BestOnly))
        return;
    if (Kind == 0) {
        Node *N = FirstNode;
        for (i = 0; i < Dimension; i++) {
            Buffer[i] = N->Id;
            N = N->Suc;
        }
    } else
        for (i = 0; i < Dimension; i++)
            Buffer[i] = BetterTour[i + 1];
    gettimeofday(&Now, 0);
    Head[0] = Kind;
    Head[1] = Trial;
    Head[2] = (long long) RecordCost;
    Head[3] = (long long) (1000.0 * (GetTime() - RecordEntryTime));
    Head[4] = (long long) Now.tv_sec * 1000 + Now.tv_usec / 1000;
    fwrite(Head, sizeof(long long), 5, File);
    fwrite(Buffer, sizeof(int), (size_t) Dimension, File);
    fflush(File);
}
'''

_LKH_EDITS = [
    ('#include "LKH.h"\n',
     '#include "LKH.h"\n' + _LKH_FUNCTION),
    ("        Cost = LinKernighan();\n        if (GetTime() - EntryTime < TimeLimit &&",
     "        Cost = LinKernighan();\n        BetaTourRecord(0, Cost, EntryTime);\n"
     "        if (GetTime() - EntryTime < TimeLimit &&"),
    ("            RecordBetterTour();\n            if ((BetterPenalty < BestPenalty ||",
     "            RecordBetterTour();\n            BetaTourRecord(1, BetterCost, EntryTime);\n"
     "            if ((BetterPenalty < BestPenalty ||"),
]

_EAX_OLD = '    printf( "%d: %d %lf\\n", fCurNumOfGen, fBestValue, fAverageValue );\n'
_EAX_NEW = r'''    {
      /* Run v25 (pilot): wall-clock time stamp of every generation and an optional wall-clock deadline. */
      struct timeval pilotNow;
      gettimeofday( &pilotNow, NULL );
      double pilotTime = (double)pilotNow.tv_sec + 1e-6 * (double)pilotNow.tv_usec;
      printf( "%d: %d %lf %.3f %d\n", fCurNumOfGen, fBestValue, fAverageValue, pilotTime, fStage );
      fflush( stdout );
      const char* pilotDeadline = getenv( "EAX_DEADLINE" );
      if( pilotDeadline != NULL && pilotTime >= atof( pilotDeadline ) ) break;
    }
'''
_EAX_HEAD = "#include <stdio.h>\n#include <stdlib.h>\n#include <sys/time.h>\n"

_EAX_NEAREST = r'''
#include <math.h>
#include <stdlib.h>
#include <vector>
#include <algorithm>
#include <utility>

/* Run v25 (pilot): the nearest cities of every city, found with a grid instead of a scan of all cities.
 * The result is the list of the original loops: ascending rounded distance and, among equal distances, the larger
 * city number first. nearDis may be NULL. Requires N > K + 1. */
static void PilotNearest( const double* x, const double* y, int N, int K, int** nearCity, int** nearDis )
{
  double xmin = x[0], xmax = x[0], ymin = y[0], ymax = y[0];
  for( int i = 1; i < N; ++i ){
    if( x[i] < xmin ) xmin = x[i];
    if( x[i] > xmax ) xmax = x[i];
    if( y[i] < ymin ) ymin = y[i];
    if( y[i] > ymax ) ymax = y[i];
  }
  double span = ( xmax - xmin > ymax - ymin ) ? xmax - xmin : ymax - ymin;
  int G = (int)sqrt( (double)N / 2.0 );
  if( G < 1 ) G = 1;
  double cell = span / (double)G;
  if( !( cell > 0.0 ) ){ cell = 1.0; G = 1; }
  std::vector<int> cx( N ), cy( N ), head( G * G + 1, 0 ), order( N );
  for( int i = 0; i < N; ++i ){
    int a = (int)( ( x[i] - xmin ) / cell ), b = (int)( ( y[i] - ymin ) / cell );
    cx[i] = a < 0 ? 0 : ( a >= G ? G - 1 : a );
    cy[i] = b < 0 ? 0 : ( b >= G ? G - 1 : b );
    ++head[ cx[i] * G + cy[i] + 1 ];
  }
  for( int c = 0; c < G * G; ++c ) head[ c + 1 ] += head[ c ];
  {
    std::vector<int> next( head.begin(), head.end() - 1 );
    for( int i = 0; i < N; ++i ) order[ next[ cx[i] * G + cy[i] ]++ ] = i;
  }
  std::vector< std::pair<int,int> > cand;
  for( int i = 0; i < N; ++i ){
    cand.clear();
    for( int r = 0; ; ++r ){
      for( int a = cx[i] - r; a <= cx[i] + r; ++a ){
        if( a < 0 || a >= G ) continue;
        for( int b = cy[i] - r; b <= cy[i] + r; ++b ){
          if( b < 0 || b >= G ) continue;
          if( abs( a - cx[i] ) != r && abs( b - cy[i] ) != r ) continue;   /* the cells of ring r only */
          for( int p = head[ a * G + b ]; p < head[ a * G + b + 1 ]; ++p ){
            int j = order[ p ];
            if( j == i ) continue;
            int d = (int)(sqrt((x[i]-x[j])*(x[i]-x[j])+(y[i]-y[j])*(y[i]-y[j]))+0.5);
            cand.push_back( std::make_pair( d, -j ) );
          }
        }
      }
      if( cx[i] - r <= 0 && cx[i] + r >= G - 1 && cy[i] - r <= 0 && cy[i] + r >= G - 1 ) break;   /* whole grid */
      if( (int)cand.size() >= K ){
        std::nth_element( cand.begin(), cand.begin() + ( K - 1 ), cand.end() );
        /* A city outside the rings 0..r is farther than r * cell from city i. */
        if( (double)r * cell > (double)cand[ K - 1 ].first + 1.0 ) break;
      }
    }
    std::partial_sort( cand.begin(), cand.begin() + K, cand.end() );
    nearCity[ i ][ 0 ] = i;
    if( nearDis != NULL ) nearDis[ i ][ 0 ] = 0;
    for( int k = 1; k <= K; ++k ){
      nearCity[ i ][ k ] = -cand[ k - 1 ].second;
      if( nearDis != NULL ) nearDis[ i ][ k ] = cand[ k - 1 ].first;
    }
  }
}
'''
_EAX_EVALUATOR_HEAD = '#ifndef __EVALUATOR__\n#include "evaluator.h"\n#endif\n'
_EAX_NORMAL_START = "  for( ci = 0; ci < Ncity; ++ci )\n  {\n    for( j3 = 0; j3 < Ncity; ++j3 ) checkedN[ j3 ] = 0;"
_EAX_LARGE_START = '  for( int i = 0; i < Ncity ; ++i )\n  {\n    if( strcmp( fType, "EUC_2D" ) == 0  ) {'
_EAX_LARGE_END = "  delete [] DisTmp;\n}"


def sha256_text(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def _replace_once(text, old, new):
    assert text.count(old) == 1, "the source does not hold the expected line exactly once: " + old[:60].strip()
    return text.replace(old, new)


def patch_lkh_findtour(text):
    """FindTour.c of LKH-3.0.14 -> the same file with the recording calls (see _LKH_FUNCTION)."""
    assert sha256_text(text) == LKH_FINDTOUR_SHA256, "FindTour.c is not the file of LKH-3.0.14"
    for old, new in _LKH_EDITS:
        text = _replace_once(text, old, new)
    return text


def patch_eax_env(text, version):
    """env.cpp of GA-EAX (version 'Normal' or 'Large') -> time stamps and deadline (see _EAX_NEW)."""
    assert sha256_text(text) == EAX_ENV_SHA256[version], "env.cpp is not the file of the pinned GA-EAX commit"
    return _EAX_HEAD + _replace_once(text, _EAX_OLD, _EAX_NEW)


def patch_eax_evaluator(text, version):
    """evaluator.cpp of GA-EAX -> the lists of nearest cities come from PilotNearest (EUC_2D, more cities than the
    list length); the original loops stay in place for every other case."""
    assert sha256_text(text) == EAX_EVALUATOR_SHA256[version], "evaluator.cpp is not the file of the pinned GA-EAX commit"
    assert text.startswith(_EAX_EVALUATOR_HEAD)
    if version == "Normal":
        assert text.count(_EAX_NORMAL_START) == 1
        start = text.index(_EAX_NORMAL_START)
        end = text.index("\n}\n", start)
        call = ('  if( strcmp( type, "EUC_2D" ) == 0 && Ncity > fNearNumMax + 1 )\n'
                "    PilotNearest( x, y, Ncity, fNearNumMax, fNearCity, NULL );\n  else\n  {\n")
        text = text[:start] + call + text[start:end] + "\n  }" + text[end:]
    else:
        assert text.count(_EAX_LARGE_START) == 1 and text.count(_EAX_LARGE_END) == 1
        start, end = text.index(_EAX_LARGE_START), text.index(_EAX_LARGE_END)
        call = ("  if( Ncity > fNearNumMax + 1 )\n"
                "    PilotNearest( x, y, Ncity, fNearNumMax, fNearCity, fEdgeDisOrder );\n  else\n  {\n")
        text = text[:start] + call + text[start:end].rstrip("\n") + "\n  }\n\n" + text[end:]
    return _EAX_EVALUATOR_HEAD + _EAX_NEAREST + text[len(_EAX_EVALUATOR_HEAD):]


def source_digest(folder, suffixes=(".cpp", ".h")):
    """One SHA-256 over the names and contents of the source files of a folder."""
    h = hashlib.sha256()
    for name in sorted(os.listdir(folder)):
        if name.endswith(suffixes):
            h.update(name.encode("ascii") + b"\0")
            with open(os.path.join(folder, name), "rb") as f:
                h.update(f.read() + b"\0")
    return h.hexdigest()


# ------------------------------------------------------------------------------------------------------ processes
def run_process(cmd, cwd, env=None, timeout=None):
    """Run a process; on a timeout it is stopped and the output printed so far is kept."""
    full_env = dict(os.environ)
    full_env.update({k: str(v) for k, v in (env or {}).items()})
    t_start = time.time()
    try:
        done = subprocess.run([str(c) for c in cmd], cwd=str(cwd), env=full_env, stdin=subprocess.DEVNULL,
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=timeout)
        out, code, timed_out = done.stdout, done.returncode, False
    except subprocess.TimeoutExpired as exc:
        out, code, timed_out = exc.stdout or b"", None, True
    return dict(stdout=out.decode("latin-1", "replace"), returncode=code, timed_out=timed_out, t_start=t_start,
                wall=time.time() - t_start)


def write_par(path, params, merge_tour_files=()):
    """LKH parameter file; every entry of merge_tour_files becomes one MERGE_TOUR_FILE line."""
    with open(path, "w") as f:
        for key, value in params.items():
            f.write(f"{key} = {value}\n")
        for name in merge_tour_files:
            f.write(f"MERGE_TOUR_FILE = {name}\n")


_PRE_RX = re.compile(r"Preprocessing time = ([0-9.]+) sec")
_TRIAL_RX = re.compile(r"^(\*?)[ \t]*(\d+):[ \t]*Cost[ \t]*=[ \t]*(-?\d+)[^\n]*?Time[ \t]*=[ \t]*([\d.]+)[ \t]*sec", re.M)


def lkh_preprocessing_s(stdout):
    m = _PRE_RX.search(stdout)
    return float(m.group(1)) if m else None


def lkh_trial_costs(stdout):
    """(trial, cost) of every trial line of a TRACE_LEVEL = 2 output (improving trials carry a star)."""
    return [(int(k), int(c)) for _, k, c, _ in _TRIAL_RX.findall(stdout)]


# ------------------------------------------------------------------------------------------------- recorded tours
def record_dtype(n):
    return np.dtype([("kind", "<i8"), ("trial", "<i8"), ("cost", "<i8"), ("lkh_ms", "<i8"), ("epoch_ms", "<i8"),
                     ("ids", "<i4", (int(n),))])


def read_records(path, n):
    """All complete records of a file written by the recording copy of LKH (a last, cut record is dropped)."""
    dtype = record_dtype(n)
    count = os.path.getsize(path) // dtype.itemsize if os.path.exists(path) else 0
    if count == 0:
        return np.zeros(0, dtype=dtype)
    return np.fromfile(path, dtype=dtype, count=count)


def tour_of(record):
    """0-based tour of a record."""
    return record["ids"].astype(np.int64) - 1


def is_permutation(tour, n):
    tour = np.asarray(tour)
    return tour.shape == (n,) and tour.min() == 0 and tour.max() == n - 1 and len(np.unique(tour)) == n


def edge_keys(tour, n):
    """Sorted undirected edge keys of a tour."""
    a = np.asarray(tour, dtype=np.int64)
    b = np.roll(a, -1)
    return np.sort(np.minimum(a, b) * n + np.maximum(a, b))


def distinct_tours(tours, n):
    """Number of different tours (as sets of undirected edges)."""
    return len({edge_keys(t, n).tobytes() for t in tours})


def common_edge_keys(tours, n):
    """Sorted keys of the undirected edges that every tour holds."""
    common = edge_keys(tours[0], n)
    for tour in tours[1:]:
        common = common[np.isin(common, edge_keys(tour, n), assume_unique=True)]
    return common


def improvements(records, start_ms):
    """[[seconds since start_ms, cost], ...] of the new best tours of one process, in time order."""
    best = records[records["kind"] == 1]
    return [[round((int(e) - start_ms) / 1000.0, 3), int(c)] for e, c in zip(best["epoch_ms"], best["cost"])]


def best_before(curve, limit):
    """Smallest cost of a [[t, cost], ...] curve at times <= limit, or None."""
    costs = [c for t, c in curve if t <= limit + 1e-9]
    return min(costs) if costs else None


def select_population(record_list, cut_ms, per_process, n, trial_tours=True):
    """Population at wall-clock time cut_ms: per process its best tour at that time and, with trial_tours, its last
    (per_process - 1) distinct trial tours that ended before that time. Returns 0-based tours, the costs recorded by
    LKH and one information dictionary per process."""
    tours, costs, info = [], [], []
    for records in record_list:
        before = records[records["epoch_ms"] <= cut_ms]
        best = before[before["kind"] == 1]
        trials = before[before["kind"] == 0]
        chosen, seen = [], set()
        if len(best):
            chosen.append(best[-1])
            seen.add(edge_keys(tour_of(best[-1]), n).tobytes())
        used_trials = []
        if trial_tours:
            for i in range(len(trials) - 1, -1, -1):
                if len(chosen) >= per_process:
                    break
                key = edge_keys(tour_of(trials[i]), n).tobytes()
                if key in seen:
                    continue
                seen.add(key)
                chosen.append(trials[i])
                used_trials.append(int(trials[i]["trial"]))
        for rec in chosen:
            tours.append(tour_of(rec))
            costs.append(int(rec["cost"]))
        info.append(dict(trials_before_cut=int(len(trials)), best_tours_before_cut=int(len(best)),
                         best_cost_at_cut=int(best[-1]["cost"]) if len(best) else None,
                         tours_used=int(len(chosen)),
                         trial_range=[min(used_trials), max(used_trials)] if used_trials else None))
    return tours, costs, info


def population_stats(tours, n, pairs=200, seed=0):
    keys = [edge_keys(t, n) for t in tours]
    union, count = np.unique(np.concatenate(keys), return_counts=True)
    rng = np.random.default_rng(seed)
    shared = []
    if len(keys) >= 2:
        for _ in range(min(pairs, len(keys) * (len(keys) - 1) // 2)):
            i, j = rng.choice(len(keys), 2, replace=False)
            shared.append(float(np.isin(keys[i], keys[j], assume_unique=True).mean()))
    return dict(tours=int(len(tours)), distinct=int(len({k.tobytes() for k in keys})),
                union_edges=int(len(union)), common_edges=int((count == len(keys)).sum()),
                pairwise_shared=float(np.mean(shared)) if shared else 1.0)


# ---------------------------------------------------------------------------------------------------------- GA-EAX
def write_eax_problem(path, xy):
    """EUC_2D instance in the layout that GA-EAX reads (keyword, colon and value separated by blanks)."""
    xy = np.asarray(xy)
    integral = np.issubdtype(xy.dtype, np.integer) or bool(np.all(xy == np.round(xy)))
    with open(path, "w") as f:
        f.write(f"NAME : instance\nTYPE : TSP\nDIMENSION : {len(xy)}\nEDGE_WEIGHT_TYPE : EUC_2D\nNODE_COORD_SECTION\n")
        if integral:
            f.writelines(f"{i} {int(x)} {int(y)}\n" for i, (x, y) in enumerate(xy, 1))
        else:
            f.writelines(f"{i} {float(x)!r} {float(y)!r}\n" for i, (x, y) in enumerate(xy, 1))
        f.write("EOF\n")


def write_population(path, tours, costs):
    with open(path, "w") as f:
        for tour, cost in zip(tours, costs):
            f.write(f"{len(tour)} {int(cost)}\n" + " ".join(map(str, (np.asarray(tour) + 1).tolist())) + " \n")


_EAX_LINE = re.compile(r"^(\d+): (-?\d+) ([0-9.eE+-]+) ([0-9.]+) (\d)$", re.M)


def parse_eax_output(stdout, launch_epoch):
    """Generation lines of the patched GA-EAX: improvements as [[seconds since launch, best], ...]."""
    rows = [(int(g), int(b), float(a), float(t), int(s)) for g, b, a, t, s in _EAX_LINE.findall(stdout)]
    curve, best = [], None
    for g, b, a, t, s in rows:
        if best is None or b < best:
            best = b
            curve.append([round(t - launch_epoch, 3), b])
    return dict(generations=rows[-1][0] if rows else None, lines=len(rows), curve=curve,
                first_line_s=round(rows[0][3] - launch_epoch, 3) if rows else None,
                last_line_s=round(rows[-1][3] - launch_epoch, 3) if rows else None,
                last_best=rows[-1][1] if rows else None, last_average=rows[-1][2] if rows else None,
                last_stage=rows[-1][4] if rows else None)


def read_eax_best(path, n):
    """Last tour of a <name>_BestSol file: (0-based tour, length reported by GA-EAX)."""
    with open(path) as f:
        tokens = f.read().split()
    assert len(tokens) >= n + 2 and len(tokens) % (n + 2) == 0, "unexpected GA-EAX tour file"
    block = tokens[-(n + 2):]
    assert int(block[0]) == n
    return np.array(block[2:], dtype=np.int64) - 1, int(block[1])


def run_eax(binary, problem_file, workdir, name, pop_size, kids=30, population_file=None, deadline_epoch=None,
            grace_s=120.0):
    """One GA-EAX run. With population_file the run starts from these tours (pop_size = their number); without it
    GA-EAX builds its own population. deadline_epoch: wall-clock time at which GA-EAX stops with its best tour.
    Returns the parsed output, the best tour (0-based) and the length reported by GA-EAX."""
    os.makedirs(workdir, exist_ok=True)
    for suffix in ("_BestSol", "_Result"):
        path = os.path.join(workdir, name + suffix)
        if os.path.exists(path):
            os.remove(path)
    cmd = [binary, "1", name, str(int(pop_size)), str(int(kids)), problem_file]
    if population_file is not None:
        cmd.append(population_file)
    env = {} if deadline_epoch is None else {"EAX_DEADLINE": f"{deadline_epoch:.3f}"}
    timeout = None if deadline_epoch is None else max(5.0, deadline_epoch - time.time()) + grace_s
    res = run_process(cmd, workdir, env=env, timeout=timeout)
    out = parse_eax_output(res["stdout"], res["t_start"])
    out.update(returncode=res["returncode"], timed_out=res["timed_out"], wall_s=float(res["wall"]),
               launch_epoch=float(res["t_start"]), deadline_s=None if deadline_epoch is None
               else float(deadline_epoch - res["t_start"]), tail=res["stdout"][-400:])
    best_file = os.path.join(workdir, name + "_BestSol")
    out["finished"] = bool(res["returncode"] == 0 and os.path.exists(best_file))
    out["stopped_by_deadline"] = bool(out["finished"] and deadline_epoch is not None and out["last_line_s"] is not None
                                      and res["t_start"] + out["last_line_s"] >= deadline_epoch - 1e-3)
    return out, best_file


# ------------------------------------------------------------------------------------------------------ statistics
def sign_test_p(wins, losses):
    """Exact two-sided sign test (ties excluded)."""
    m = wins + losses
    if m == 0:
        return 1.0
    k = min(wins, losses)
    return float(min(1.0, 2.0 * sum(comb(m, i) for i in range(k + 1)) / 2.0 ** m))


def holm(pvalues):
    """Holm-adjusted p-values, in the order given."""
    order = sorted(range(len(pvalues)), key=lambda i: pvalues[i])
    adjusted, running = [0.0] * len(pvalues), 0.0
    for rank, i in enumerate(order):
        running = max(running, min(1.0, (len(pvalues) - rank) * float(pvalues[i])))
        adjusted[i] = running
    return adjusted


def paired_stats(values, bootstrap_seed, replicates=5000):
    """Paired percentage differences (first minus second arm): wins are negative values (the first arm is shorter)."""
    x = np.asarray(values, dtype=np.float64)
    assert x.ndim == 1 and len(x) and np.isfinite(x).all()
    wins, losses = int((x < -1e-12).sum()), int((x > 1e-12).sum())
    rng = np.random.default_rng(bootstrap_seed)
    means = x[rng.integers(0, len(x), size=(replicates, len(x)))].mean(axis=1)
    low, high = np.quantile(means, [0.025, 0.975])
    return dict(instances=int(len(x)), win=wins, tie=int(len(x) - wins - losses), loss=losses,
                mean_delta_pct=float(x.mean()), median_delta_pct=float(np.median(x)),
                ci95_low_pct=float(low), ci95_high_pct=float(high), sign_p=sign_test_p(wins, losses))


In [ ]:
%%writefile unionx_v27.py
"""Run v27 (pilot): exact merging of tours on their union graph. Helper module embedded in notebook 29.

The union graph of a set of tours has the n nodes and every edge that at least one of the tours uses. A tour of the
union graph can combine parts of the given tours. This module finds the shortest tour of the union graph exactly,
optionally after adding extra edges, by branch and cut with SCIP (through PySCIPOpt):

  * Reduction. Two rules hold for every tour of the graph and are applied until nothing changes: a node with two
    remaining edges uses both (the edges are fixed), and a node with two fixed edges uses no other edge (its other
    edges are removed). An edge that would close a path of fixed edges into a cycle of fewer than n nodes is
    removed as well.
  * Branch and cut. One binary variable per remaining free edge, one degree equation per node with fewer than two
    fixed edges, and subtour elimination constraints added by a constraint handler: whenever a solution of the LP
    relaxation (at the thresholds 1e-6, 0.3, 0.5 and 0.7) or an integral solution falls apart into several
    connected components, the handler adds x(delta(S)) >= 2 for every component S that violates it. The fixed paths
    are contracted, so a component is a set of paths.
  * Start. The shortest given tour is passed to SCIP as its first solution, so the result is never longer.

Extra edges are taken from the graph of the BetaTour model and must be absent from the union: either the edges with
the highest predicted probability or, as a control with the same number, the shortest ones.

The notebook runs every solve in its own process through the command line of this module (``solve IN OUT``) with a
wall-clock deadline. Pure NumPy apart from PySCIPOpt.
"""

from __future__ import annotations

import json
import math
import sys
import time
from collections import deque
from itertools import permutations

import numpy as np

VERSION = "v27-unionx-1"
THRESHOLDS = (1e-6, 0.3, 0.5, 0.7)


# ------------------------------------------------------------------------------------------------------ edges
def keys_of(a, b, n):
    a = np.asarray(a, dtype=np.int64)
    b = np.asarray(b, dtype=np.int64)
    return np.minimum(a, b) * int(n) + np.maximum(a, b)


def tour_keys(tour):
    """Sorted keys of the n undirected edges of a tour (a permutation of 0..n-1)."""
    t = np.asarray(tour, dtype=np.int64)
    return np.unique(keys_of(t, np.roll(t, -1), len(t)))


def union_keys(tours, n):
    return np.unique(np.concatenate([keys_of(np.asarray(t), np.roll(np.asarray(t), -1), n) for t in tours]))


def edge_length(xy, u, v, ew="EUC_2D"):
    """Integer TSPLIB lengths, computed as tsplab.edge_len does."""
    d = np.sqrt(((np.asarray(xy, dtype=np.float64)[u] - np.asarray(xy, dtype=np.float64)[v]) ** 2).sum(-1))
    if ew == "EUC_2D":
        return np.floor(d + 0.5).astype(np.int64)
    if ew == "CEIL_2D":
        return np.ceil(d).astype(np.int64)
    raise ValueError(f"unsupported EDGE_WEIGHT_TYPE {ew}")


def tour_length(xy, tour, ew="EUC_2D"):
    t = np.asarray(tour, dtype=np.int64)
    return int(edge_length(xy, t, np.roll(t, -1), ew).sum())


def is_permutation(tour, n):
    t = np.asarray(tour)
    return t.shape == (n,) and len(np.unique(t)) == n and t.min() == 0 and t.max() == n - 1


# --------------------------------------------------------------------------------------------------- reduction
def reduce_graph(n, keys):
    """Fixed and removed edges that hold for every tour of the graph with the given edge keys.

    Returns status per edge (0 free, 1 fixed, 2 removed), the number of fixed edges per node, the root of the fixed
    path of every node and whether the fixed edges already form a tour. Raises ValueError if the graph has no tour
    by these rules (it cannot happen for the union of tours)."""
    keys = np.asarray(keys, dtype=np.int64)
    eu, ev = keys // n, keys % n
    m = len(keys)
    adj = [[] for _ in range(n)]
    for e in range(m):
        adj[eu[e]].append(e)
        adj[ev[e]].append(e)
    status = np.zeros(m, dtype=np.int8)
    deg = np.array([len(a) for a in adj], dtype=np.int64)
    fdeg = np.zeros(n, dtype=np.int64)
    other = np.arange(n, dtype=np.int64)          # for an end of a fixed path: the other end
    parent = np.arange(n, dtype=np.int64)
    size = np.ones(n, dtype=np.int64)
    closed = False
    queue = deque(range(n))

    def find(x):
        r = x
        while parent[r] != r:
            r = parent[r]
        while parent[x] != r:
            parent[x], x = r, parent[x]
        return r

    def remove(e):
        status[e] = 2
        for x in (eu[e], ev[e]):
            deg[x] -= 1
            queue.append(int(x))

    def fix(e):
        nonlocal closed
        a, b = int(eu[e]), int(ev[e])
        if fdeg[a] >= 2 or fdeg[b] >= 2:
            raise ValueError("no tour: a node would get three fixed edges")
        status[e] = 1
        oa, ob = int(other[a]), int(other[b])
        ra, rb = find(a), find(b)
        if ra == rb:
            if size[ra] != n:
                raise ValueError("no tour: the fixed edges close a subtour")
            closed = True
        else:
            if size[ra] < size[rb]:
                ra, rb = rb, ra
            parent[rb] = ra
            size[ra] += size[rb]
            other[oa], other[ob] = ob, oa
            if size[ra] < n:
                for e2 in adj[oa]:
                    if status[e2] == 0 and (eu[e2] == ob or ev[e2] == ob):
                        remove(e2)
        fdeg[a] += 1
        fdeg[b] += 1
        queue.extend((a, b, oa, ob))

    while queue:
        v = queue.popleft()
        if fdeg[v] == 2:
            for e in adj[v]:
                if status[e] == 0:
                    remove(e)
        elif deg[v] == 2:
            for e in adj[v]:
                if status[e] == 0:
                    fix(e)
        if deg[v] < 2:
            raise ValueError("no tour: a node has fewer than two edges")
    roots = np.array([find(v) for v in range(n)], dtype=np.int64)
    return dict(status=status, fdeg=fdeg, roots=roots, closed=bool(closed))


def tour_from_edges(n, keys, chosen):
    """The tour formed by the chosen edges (boolean mask over keys); raises if they do not form one tour."""
    keys = np.asarray(keys, dtype=np.int64)[np.asarray(chosen, dtype=bool)]
    nb = np.full((n, 2), -1, dtype=np.int64)
    count = np.zeros(n, dtype=np.int64)
    for k in keys:
        a, b = int(k // n), int(k % n)
        if count[a] >= 2 or count[b] >= 2:
            raise ValueError("a node has more than two chosen edges")
        nb[a, count[a]] = b
        nb[b, count[b]] = a
        count[a] += 1
        count[b] += 1
    if not np.all(count == 2):
        raise ValueError("the chosen edges are not 2-regular")
    tour = np.empty(n, dtype=np.int64)
    prev, cur = -1, 0
    for i in range(n):
        tour[i] = cur
        nxt = nb[cur, 0] if nb[cur, 0] != prev else nb[cur, 1]
        prev, cur = cur, nxt
    if cur != 0 or len(np.unique(tour)) != n:
        raise ValueError("the chosen edges form more than one cycle")
    return tour


# ------------------------------------------------------------------------------------------------ branch and cut
def _scip():
    from pyscipopt import Conshdlr, Model, SCIP_RESULT, quicksum
    return Conshdlr, Model, SCIP_RESULT, quicksum


def solve_exact(xy, keys, start_tour, time_limit, ew="EUC_2D"):
    """Shortest tour of the graph with the given edge keys, by branch and cut with SCIP within time_limit seconds.

    start_tour must be a tour of the graph. Returns (tour, info); the tour is the best one found (the start tour if
    nothing shorter was found in time) and info["optimal"] says whether SCIP proved it shortest."""
    t_begin = time.perf_counter()
    xy = np.asarray(xy)
    n = len(xy)
    keys = np.asarray(keys, dtype=np.int64)
    assert np.all(np.diff(keys) > 0), "edge keys must be sorted and unique"
    start = np.asarray(start_tour, dtype=np.int64)
    start_keys = tour_keys(start)
    assert is_permutation(start, n) and np.isin(start_keys, keys).all(), "the start tour is not a tour of the graph"
    length = edge_length(xy, keys // n, keys % n, ew)
    red = reduce_graph(n, keys)
    status, fdeg, roots = red["status"], red["fdeg"], red["roots"]
    info = dict(version=VERSION, nodes=int(n), edges=int(len(keys)), fixed=int((status == 1).sum()),
                removed=int((status == 2).sum()), free=int((status == 0).sum()), core=int((fdeg < 2).sum()),
                start_length=int(tour_length(xy, start, ew)), start_has_all_fixed=None,
                start_avoids_removed=None)
    pos_of = {int(k): i for i, k in enumerate(keys)}
    in_start = np.zeros(len(keys), dtype=bool)
    in_start[[pos_of[int(k)] for k in start_keys]] = True
    # Every tour of the graph holds the fixed edges and none of the removed ones; the start tour must too.
    info["start_has_all_fixed"] = bool(np.all(in_start[status == 1]))
    info["start_avoids_removed"] = bool(not np.any(in_start[status == 2]))
    assert info["start_has_all_fixed"] and info["start_avoids_removed"], "reduction contradicts the start tour"
    info["reduce_s"] = round(time.perf_counter() - t_begin, 4)
    free = np.flatnonzero(status == 0)
    if red["closed"] or len(free) == 0:
        tour = tour_from_edges(n, keys, status == 1)
        info.update(status="fixed", optimal=True, solve_s=0.0, bnb_nodes=0, cuts=0, gap=0.0,
                    dual_bound=None, length=int(tour_length(xy, tour, ew)), time_to_best_s=0.0,
                    solve_start_epoch=None)
        return tour, info
    Conshdlr, Model, SCIP_RESULT, quicksum = _scip()
    fu, fv = keys[free] // n, keys[free] % n
    core = np.flatnonzero(fdeg < 2)
    _, sup = np.unique(roots[core], return_inverse=True)
    sup_of = np.full(n, -1, dtype=np.int64)
    sup_of[core] = sup
    ea, eb = sup_of[fu], sup_of[fv]
    assert np.all(ea >= 0) and np.all(eb >= 0)
    nsup = int(sup.max()) + 1

    class Subtour(Conshdlr):
        def components(self, vals, thr):
            parent = np.arange(nsup)

            def find(x):
                r = x
                while parent[r] != r:
                    r = parent[r]
                while parent[x] != r:
                    parent[x], x = r, parent[x]
                return r
            for j in np.flatnonzero(vals > thr):
                ra, rb = find(ea[j]), find(eb[j])
                if ra != rb:
                    parent[ra] = rb
            return np.array([find(i) for i in range(nsup)])

        def values(self, solution):
            return np.array([self.model.getSolVal(solution, v) for v in xvars])

        def connected(self, vals):
            return len(np.unique(self.components(vals, 0.5))) == 1

        def cut(self, vals, thresholds, force):
            if self.tvars is None:
                self.tvars = [self.model.getTransformedVar(v) for v in xvars]
            added = 0
            for thr in thresholds:
                lab = self.components(vals, thr)
                labels = np.unique(lab)
                if len(labels) == 1:
                    continue
                la, lb = lab[ea], lab[eb]
                for label in labels:
                    cross = np.flatnonzero((la == label) != (lb == label))
                    if vals[cross].sum() >= 2.0 - 1e-6:
                        continue
                    row = self.model.createEmptyRowUnspec(name="sec", lhs=2.0, rhs=None, local=False,
                                                          modifiable=False, removable=True)
                    self.model.cacheRowExtensions(row)
                    for j in cross:
                        self.model.addVarToRow(row, self.tvars[j], 1.0)
                    self.model.flushRowExtensions(row)
                    self.model.addCut(row, forcecut=force)
                    self.model.releaseRow(row)
                    added += 1
                if added:
                    break
            self.ncuts += added
            return added

        def conscheck(self, constraints, solution, checkintegrality, checklprows, printreason, completely, **kw):
            ok = self.connected(self.values(solution))
            return {"result": SCIP_RESULT.FEASIBLE if ok else SCIP_RESULT.INFEASIBLE}

        def consenfolp(self, constraints, nusefulconss, solinfeasible):
            vals = self.values(None)
            if self.connected(vals):
                return {"result": SCIP_RESULT.FEASIBLE}
            added = self.cut(vals, (0.5,), True)
            return {"result": SCIP_RESULT.SEPARATED if added else SCIP_RESULT.INFEASIBLE}

        def consenfops(self, constraints, nusefulconss, solinfeasible, objinfeasible):
            vals = self.values(None)
            return {"result": SCIP_RESULT.FEASIBLE if self.connected(vals) else SCIP_RESULT.SOLVELP}

        def conssepalp(self, constraints, nusefulconss):
            added = self.cut(self.values(None), THRESHOLDS, False)
            return {"result": SCIP_RESULT.SEPARATED if added else SCIP_RESULT.DIDNOTFIND}

        def conslock(self, constraint, locktype, nlockspos, nlocksneg):
            for v in xvars:
                self.model.addVarLocks(v, nlockspos + nlocksneg, nlockspos + nlocksneg)

    model = Model()
    model.hideOutput()
    xvars = [model.addVar(vtype="B", obj=float(length[j])) for j in free]
    incident = [[] for _ in range(n)]
    for i in range(len(free)):
        incident[fu[i]].append(xvars[i])
        incident[fv[i]].append(xvars[i])
    for v in core:
        model.addCons(quicksum(incident[v]) == 2 - int(fdeg[v]))
    handler = Subtour()
    handler.tvars, handler.ncuts = None, 0
    model.includeConshdlr(handler, "subtour", "subtour elimination on the contracted union graph", sepapriority=1,
                          enfopriority=-1, chckpriority=-1, sepafreq=1, propfreq=-1, eagerfreq=-1, maxprerounds=0,
                          delaysepa=False, delayprop=False, needscons=True)
    model.addPyCons(model.createCons(handler, "subtour", initial=False, separate=True, enforce=True, check=True,
                                     propagate=False))
    first = model.createSol()
    for i, j in enumerate(free):
        model.setSolVal(first, xvars[i], 1.0 if in_start[j] else 0.0)
    accepted = model.addSol(first, free=True)
    info["build_s"] = round(time.perf_counter() - t_begin - info["reduce_s"], 4)
    left = float(time_limit) - (time.perf_counter() - t_begin)
    if left <= 0.05:
        info.update(status="no time", optimal=False, solve_s=0.0, bnb_nodes=0, cuts=0, gap=None, dual_bound=None,
                    length=info["start_length"], time_to_best_s=None, start_accepted=bool(accepted),
                    solve_start_epoch=None)
        return start.copy(), info
    model.setParam("limits/time", left)
    info["solve_start_epoch"] = time.time()
    t_solve = time.perf_counter()
    model.optimize()
    solve_s = time.perf_counter() - t_solve
    best = model.getBestSol()
    chosen = (status == 1).copy()
    values = np.array([model.getSolVal(best, v) for v in xvars])
    chosen[free[values > 0.5]] = True
    tour = tour_from_edges(n, keys, chosen)
    scip_status = model.getStatus()
    forced_length = int(length[status == 1].sum())
    info.update(status=str(scip_status), optimal=bool(scip_status == "optimal"), solve_s=round(solve_s, 4),
                bnb_nodes=int(model.getNNodes()), cuts=int(handler.ncuts), gap=float(model.getGap()),
                dual_bound=float(model.getDualbound()) + forced_length, length=int(tour_length(xy, tour, ew)),
                objective=float(model.getObjVal()) + forced_length, time_to_best_s=float(model.getSolTime(best)),
                start_accepted=bool(accepted), scip_version=str(model.version()))
    assert info["length"] == round(info["objective"]), "the tour length differs from the SCIP objective"
    assert info["length"] <= info["start_length"], "the result is longer than the start tour"
    return tour, info


# ------------------------------------------------------------------------------------------------- extra edges
def probability_table(prob, nb):
    """Undirected edge keys of the model graph and their probability (maximum over the two directions); as
    marking.probability_table of run v23. prob[i, j] belongs to the edge from i to its neighbour nb[i, j]."""
    nb = np.asarray(nb, dtype=np.int64)
    prob = np.asarray(prob, dtype=np.float64)
    assert nb.ndim == 2 and prob.shape == nb.shape and np.all(nb >= 0)
    n, k = nb.shape
    keys = keys_of(np.repeat(np.arange(n, dtype=np.int64), k), nb.reshape(-1), n)
    order = np.argsort(keys, kind="stable")
    keys = keys[order]
    unique, start = np.unique(keys, return_index=True)
    return unique, np.maximum.reduceat(prob.reshape(-1)[order], start)


def extra_edges(table, union, count, by, xy=None, ew="EUC_2D"):
    """`count` edges of the model graph that are absent from the union: by="probability" takes the most probable
    ones, by="length" the shortest ones; ties are broken by the edge key. Returns sorted keys."""
    keys, prob = table
    n = None if xy is None else len(xy)
    novel = ~np.isin(keys, union)
    cand, p = keys[novel], prob[novel]
    if by == "probability":
        order = np.lexsort((cand, -p))
    elif by == "length":
        assert xy is not None
        order = np.lexsort((cand, edge_length(xy, cand // n, cand % n, ew)))
    else:
        raise ValueError(by)
    return np.sort(cand[order[:int(count)]])


# ------------------------------------------------------------------------------------------------- small checks
def brute_force(xy, keys, ew="EUC_2D"):
    """Shortest tour of a small graph by enumeration (node 0 first, one direction); None if there is none."""
    n = len(xy)
    allowed = set(np.asarray(keys, dtype=np.int64).tolist())
    best = None
    for rest in permutations(range(1, n)):
        if rest[0] > rest[-1]:
            continue
        t = (0,) + rest
        if all(min(t[i], t[(i + 1) % n]) * n + max(t[i], t[(i + 1) % n]) in allowed for i in range(n)):
            length = tour_length(xy, t, ew)
            if best is None or length < best[0]:
                best = (length, np.array(t, dtype=np.int64))
    return best


# ------------------------------------------------------------------------------------------------- command line
def main(argv):
    """python unionx_v27.py solve IN.npz OUT.npz

    IN holds xy, keys, start (tour), deadline (epoch seconds) and ew; OUT gets the tour and the info as JSON."""
    assert len(argv) == 3 and argv[0] == "solve", main.__doc__
    with np.load(argv[1], allow_pickle=False) as z:
        xy, keys, start = z["xy"], z["keys"], z["start"]
        deadline, ew = float(z["deadline"]), str(z["ew"])
    tour, info = solve_exact(xy, keys, start, deadline - time.time(), ew)
    info["finished_epoch"] = time.time()
    tmp = argv[2] + ".tmp.npz"
    np.savez_compressed(tmp, tour=tour.astype(np.int32), info=np.array(json.dumps(info)))
    import os
    os.replace(tmp, argv[2])
    return 0


if __name__ == "__main__":
    sys.exit(main(sys.argv[1:]))


In [ ]:
import hashlib
import importlib.util
import json
import math
import platform
import shutil
import subprocess
import sys
import tarfile
import threading
import time
import traceback
import urllib.request
import uuid
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

if importlib.util.find_spec("pandas") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas"], check=True)
try:
    import pyscipopt
    assert pyscipopt.__version__ == PYSCIPOPT_VERSION
except (ImportError, AssertionError):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"pyscipopt=={PYSCIPOPT_VERSION}"], check=True)
    importlib.invalidate_caches()
    for loaded in [m for m in sys.modules if m == "pyscipopt" or m.startswith("pyscipopt.")]:
        del sys.modules[loaded]
    import pyscipopt
    assert pyscipopt.__version__ == PYSCIPOPT_VERSION, \
        f"PySCIPOpt {PYSCIPOPT_VERSION} kuruldu ama eski sürüm yüklü kaldı; Çalışma zamanı > Oturumu yeniden başlat deyip yeniden çalıştırın."
SCIP_VERSION = str(pyscipopt.Model().version())

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import numpy as np
import pandas as pd
import torch

CPU_COUNT = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
cpu_max = Path("/sys/fs/cgroup/cpu.max")
if cpu_max.exists():
    quota, period = cpu_max.read_text().split()[:2]
    if quota != "max":
        CPU_COUNT = min(CPU_COUNT, max(1, math.floor(int(quota) / int(period))))
assert CPU_COUNT >= W, f"{W} süreç için en az {W} vCPU gerekli; görünen: {CPU_COUNT}"
if REQUIRE_12_VCPUS:
    assert CPU_COUNT >= 2 * W, f"Önceki koşulardaki düzen için 12 vCPU gerekli (Yüksek RAM seçin); görünen: {CPU_COUNT}"


def cpu_model_name():
    path = Path("/proc/cpuinfo")
    if path.exists():
        for line in path.read_text().splitlines():
            if line.lower().startswith("model name"):
                return line.split(":", 1)[1].strip()
    return platform.processor() or "unknown"


def gpu_name():
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                             stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True, timeout=30)
        return out.stdout.strip().splitlines()[0] if out.returncode == 0 and out.stdout.strip() else "none"
    except Exception:
        return "none"


CPU_MODEL = cpu_model_name()
GPU_NAME = gpu_name()
WORK_ROOT = Path(os.environ.get("BETATOUR_V27_WORK", "/content/betatour_v27_work"))
WORK_ROOT.mkdir(parents=True, exist_ok=True)

# Depo: run v5-v26 ile aynı sabit commit (örnek üreticisi, BetaTour modeli ve eğitilmiş model dosyası).
if TEST_MODE and os.environ.get("BETATOUR_V27_REPO"):
    REPO = Path(os.environ["BETATOUR_V27_REPO"])
else:
    REPO = WORK_ROOT / "repo"
    if not (REPO / ".git").exists():
        subprocess.run(["git", "clone", "--quiet",
                        "https://github.com/kurbankotan/betatour-tsp-lkh-candidates.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "--quiet", "origin", REPO_COMMIT], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REPO_COMMIT], check=True)
    assert subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip() == REPO_COMMIT
sys.path.insert(0, str(REPO / "src"))
import tsplab as T
import tsp_model as M

# Protokol koşudan önce GitHub'da olmalı: main dalındaki protokol dosyası bu defterin kurulduğu protokolle bayt bayt
# aynı değilse kurulum durur. Dosyayı son değiştiren commit ve zamanı manifeste yazılır (ilk örnekten önce).
PROTOCOL = {"path": PROTOCOL_PATH, "sha256": PROTOCOL_SHA256, "github_head": None, "protocol_commit": None,
            "protocol_commit_time": None, "notebook_on_github": None, "checked_utc": None}
if REQUIRE_PUSHED_PROTOCOL:
    subprocess.run(["git", "-C", str(REPO), "fetch", "--quiet", "origin", "main"], check=True)
    github_head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "FETCH_HEAD"], text=True).strip()
    shown = subprocess.run(["git", "-C", str(REPO), "show", f"{github_head}:{PROTOCOL_PATH}"],
                           stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
    assert shown.returncode == 0, \
        f"Protokol GitHub'da yok ({PROTOCOL_PATH}). Önce depoyu commit ve push edin, sonra defteri yeniden çalıştırın."
    assert hashlib.sha256(shown.stdout).hexdigest() == PROTOCOL_SHA256, \
        "GitHub'daki protokol bu defterin protokolüyle aynı değil. Protokol değiştiyse defter yeniden kurulmalıdır."
    notebook_file = "notebooks/29_BetaTour_v27_union_exact_pilot.ipynb"
    pushed = subprocess.run(["git", "-C", str(REPO), "show", f"{github_head}:{notebook_file}"],
                            stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
    assert pushed.returncode == 0 and NOTEBOOK_BUILD.encode() in pushed.stdout, \
        "GitHub'daki defter bu defterle aynı kurulum değil. Depodaki defteri push edip onu çalıştırın."
    last = subprocess.check_output(["git", "-C", str(REPO), "log", "-1", "--format=%H %cI", github_head, "--",
                                    PROTOCOL_PATH], text=True).split()
    PROTOCOL.update(github_head=github_head, protocol_commit=last[0], protocol_commit_time=last[1],
                    notebook_on_github=True, checked_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()))
    print("Protokol GitHub'da:", PROTOCOL_PATH, "| commit", last[0][:10], last[1])

RAM_GB = T.env_info().get("ram_gb")
assert RAM_GB is None or RAM_GB >= MIN_RAM_GB, \
    f"Yüksek RAM gerekli (GA-EAX süreçleri için): görünen {RAM_GB} GB, gereken en az {MIN_RAM_GB} GB"

# Bu deneyin iki yardımcı modülü (önceki iki hücrenin yazdığı dosyalar): run v26'nın modülü (değiştirilmeden) ve
# birleşim grafında tam çözüm modülü. Dosya sonundaki satır sonu sayısı arayüze göre değişebildiği için tek satır
# sonuna indirilir; içerik depodaki modüllerle bayt bayt aynı olmalıdır.
def checked_module(file_name, expected):
    path = Path(file_name).resolve()
    assert path.exists(), f"Önce {file_name} dosyasını yazan hücreyi çalıştırın."
    data = path.read_bytes().rstrip(b"\n") + b"\n"
    digest = hashlib.sha256(data).hexdigest()
    assert digest == expected, f"{file_name} beklenen modülle aynı değil; o hücreyi değiştirmeden çalıştırın."
    if path.read_bytes() != data:
        path.write_bytes(data)
    return path, digest


CODE_PATH, CODE_SHA256 = checked_module("byprod_v26.py", CODE_SHA256_EXPECTED)
UNION_PATH, UNION_SHA256 = checked_module("unionx_v27.py", UNION_SHA256_EXPECTED)
sys.path.insert(0, str(CODE_PATH.parent))
import byprod_v26 as B
import unionx_v27 as U

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert TEST_MODE or DEVICE.type == "cuda", "GPU gerekli (L4): BetaTour çıkarımı GPU'da yapılır."
checkpoint_path = REPO / "checkpoints" / "betatour_v2.pt"
assert hashlib.sha256(checkpoint_path.read_bytes()).hexdigest() == CHECKPOINT_SHA256, \
    "Model dosyası beklenen SHA256 değeriyle eşleşmiyor."
MODEL, CK = M.load_model(str(checkpoint_path), DEVICE)   # sabitlenmiş, güvenilen depo
assert MODEL.cfg.get("use_ensemble") is True

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/BetaTour") / EXPERIMENT_NAME
else:
    OUTPUT_ROOT = WORK_ROOT / "results" / EXPERIMENT_NAME
for sub in ["instances", "tours", "stages", "errors", "summary", "code"]:
    (OUTPUT_ROOT / sub).mkdir(parents=True, exist_ok=True)
shutil.copy2(CODE_PATH, OUTPUT_ROOT / "code" / "byprod_v26.py")
shutil.copy2(UNION_PATH, OUTPUT_ROOT / "code" / "unionx_v27.py")


def run_build(cmd, cwd, what):
    build = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, timeout=1800)
    if build.returncode != 0:
        raise RuntimeError(what + " derlemesi başarısız:\n" + build.stdout[-3000:])


def findtour_matches(folder):
    path = Path(folder) / "SRC" / "FindTour.c"
    return path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == B.LKH_FINDTOUR_SHA256


def get_lkh_3014():
    """Değiştirilmemiş LKH-3.0.14. Resmi kaynak; ulaşılamazsa ya da FindTour.c beklenen dosya değilse SHA256
    denetimli sabit kopya (run v21-v23 ve v25 bu kopyayla koştu)."""
    if TEST_MODE and os.environ.get("BETATOUR_V27_LKH"):
        assert findtour_matches(Path(os.environ["BETATOUR_V27_LKH"]).parent), "yerel LKH kaynağı 3.0.14 değil"
        return os.environ["BETATOUR_V27_LKH"], "local test binary"
    root = WORK_ROOT / "lkh"
    binary = root / "LKH-3.0.14" / "LKH"
    marker = root / "source.txt"
    if binary.exists() and marker.exists() and findtour_matches(root / "LKH-3.0.14"):
        return str(binary), marker.read_text().strip()
    root.mkdir(parents=True, exist_ok=True)
    options = [
        ("official", "https://webhotel4.ruc.dk/~keld/research/LKH-3/LKH-3.0.14.tgz", None, 12),
        ("pinned copy", "https://raw.githubusercontent.com/ZhekaiChen/ALCO/main/third_party/LKH3/LKH-3.0.14.tar",
         "4c4b9e6a77e50a1eb1eb2c8cb384ed5ef64e0dc88fdbf37bd693798469e1cf33", 90),
    ]
    last_error = None
    for label, url, expected_sha, timeout in options:
        archive = root / ("LKH-3.0.14.tar" if url.endswith(".tar") else "LKH-3.0.14.tgz")
        try:
            print("LKH kaynağı:", label, url)
            request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(request, timeout=timeout) as response, archive.open("wb") as f:
                while True:
                    block = response.read(1024 * 1024)
                    if not block:
                        break
                    f.write(block)
            if expected_sha:
                got = hashlib.sha256(archive.read_bytes()).hexdigest()
                assert got == expected_sha, f"LKH kopyası SHA256 uyuşmazlığı: {got}"
            shutil.rmtree(root / "LKH-3.0.14", ignore_errors=True)
            with tarfile.open(archive, "r:*") as tar:
                try:
                    tar.extractall(root, filter="data")
                except TypeError:
                    tar.extractall(root)
            assert (root / "LKH-3.0.14" / "Makefile").exists(), "LKH kaynak klasörü eksik"
            assert findtour_matches(root / "LKH-3.0.14"), "FindTour.c beklenen dosya değil (kayıt yaması uygulanamaz)"
            run_build(["make", "-j", str(min(4, CPU_COUNT))], root / "LKH-3.0.14", "LKH")
            assert binary.exists(), "LKH ikilisi oluşmadı"
            marker.write_text(f"{label}: {url}")
            return str(binary), f"{label}: {url}"
        except Exception as exc:
            last_error = exc
            print(f"{label} başarısız: {exc}")
    raise RuntimeError(f"LKH 3.0.14 indirilemedi veya derlenemedi: {last_error}")


def build_recording_lkh(plain_binary):
    """LKH'nin kayıt eden kopyası: FindTour.c'ye iki çağrı ve bir işlev eklenir (byprod_v26.patch_lkh_findtour).
    Arama durumu değişmez; öz sınama bunu bu makinede denetler. BETATOUR_BEST_ONLY ortam değişkeni verilirse
    yalnızca yeni en iyi turlar yazılır (LKH'nin kendi tur birleştirmesini yapan süreçlerde kullanılır)."""
    source = Path(plain_binary).parent
    dest = WORK_ROOT / "lkh_rec" / "LKH-3.0.14"
    binary, marker = dest / "LKH", dest / "patched.txt"
    if binary.exists() and marker.exists() and marker.read_text() == B.VERSION:
        return str(binary)
    shutil.rmtree(dest.parent, ignore_errors=True)
    shutil.copytree(source, dest, ignore=shutil.ignore_patterns("OBJ", "LKH", "*.o", "*.tgz", "*.tar"))
    (dest / "SRC" / "OBJ").mkdir(exist_ok=True)
    path = dest / "SRC" / "FindTour.c"
    path.write_text(B.patch_lkh_findtour(path.read_text()))
    run_build(["make", "-j", str(min(4, CPU_COUNT))], dest, "kayıt eden LKH")
    assert binary.exists(), "kayıt eden LKH ikilisi oluşmadı"
    marker.write_text(B.VERSION)
    return str(binary)


def get_eax():
    """GA-EAX (Nagata ve Kobayashi, 2013): yazarın GitHub deposu, sabit commit, MIT lisansı. İki sürüm de
    (Normal, Large) iki yamayla derlenir: env.cpp'ye zaman damgası ve duvar saati sınırı (patch_eax_env),
    evaluator.cpp'ye en yakın şehir listelerinin ızgarayla bulunması (patch_eax_evaluator; aynı listeler, 50.000
    düğümde yaklaşık bir dakika yerine bir saniyeden az). Öz sınama için ikinci yamasız birer kopya da derlenir."""
    if TEST_MODE and os.environ.get("BETATOUR_V27_EAX_SRC"):
        source, label = Path(os.environ["BETATOUR_V27_EAX_SRC"]), "local test clone"
    else:
        source = WORK_ROOT / "ga_eax_src"
        if not (source / ".git").exists():
            shutil.rmtree(source, ignore_errors=True)
            subprocess.run(["git", "clone", "--quiet", "https://github.com/nagata-yuichi/GA-EAX.git", str(source)],
                           check=True)
        subprocess.run(["git", "-C", str(source), "checkout", "--quiet", B.EAX_COMMIT], check=True)
        assert subprocess.check_output(["git", "-C", str(source), "rev-parse", "HEAD"], text=True).strip() == B.EAX_COMMIT
        label = "https://github.com/nagata-yuichi/GA-EAX @ " + B.EAX_COMMIT
    binaries, plain = {}, {}
    for version in ("Normal", "Large"):
        folder = source / "GA_EAX_1.0" / version
        assert B.source_digest(str(folder)) == B.EAX_TREE_SHA256[version], \
            f"GA-EAX kaynağı ({version}) sabitlenen commit ile aynı değil"
        for grid, target in ((True, binaries), (False, plain)):
            dest = WORK_ROOT / "ga_eax_build" / (version.lower() + ("" if grid else "_plain"))
            binary, marker = dest / "eax", dest / "patched.txt"
            if not (binary.exists() and marker.exists() and marker.read_text() == B.VERSION):
                shutil.rmtree(dest, ignore_errors=True)
                dest.mkdir(parents=True)
                for path in folder.iterdir():
                    if path.suffix in {".cpp", ".h"}:
                        shutil.copy2(path, dest / path.name)
                env_file = dest / "env.cpp"
                env_file.write_text(B.patch_eax_env(env_file.read_text(), version))
                if grid:
                    evaluator_file = dest / "evaluator.cpp"
                    evaluator_file.write_text(B.patch_eax_evaluator(evaluator_file.read_text(), version))
                run_build(["g++", "-O3", "-w", "-o", "eax"] + B.EAX_SOURCES + ["-lm"], dest, "GA-EAX " + version)
                assert binary.exists(), "GA-EAX ikilisi oluşmadı"
                marker.write_text(B.VERSION)
            target[version] = str(binary)
    return binaries, plain, label


LKH, LKH_SOURCE = get_lkh_3014()
LKH_REC = build_recording_lkh(LKH)
EAX, EAX_PLAIN, EAX_SOURCE = get_eax()

HARDWARE = {
    "gpu": GPU_NAME,
    "vcpus_visible": CPU_COUNT,
    "cpu_model": CPU_MODEL,
    "ram_gb": RAM_GB,
    "python": platform.python_version(),
    "numpy": np.__version__,
    "torch": torch.__version__,
    "pyscipopt": pyscipopt.__version__,
    "scip": SCIP_VERSION,
}
MANIFEST = {
    "schema": 1,
    "experiment_name": EXPERIMENT_NAME,
    "test_mode": TEST_MODE,
    "design": {"synthetic": [list(x) for x in SYNTHETIC], "decision_groups": DECISION_GROUPS,
               "candidate_k": K_LIST, "graph_k": K_GRAPH, "split": round(float(SPLIT), 6), "per_process": PER_PROCESS,
               "extra_fraction": EXTRA_FRACTION, "merge_seeds": MERGE_SEEDS, "eax_kids": EAX_KIDS,
               "eax_normal_max_n": EAX_NORMAL_MAX_N, "stop_pad_s": STOP_PAD_S, "exact_pad_s": EXACT_PAD_S},
    "protocol": PROTOCOL,
    "seeds": SEEDS,
    "workers": W,
    "repo_commit": REPO_COMMIT,
    "code_version": B.VERSION,
    "code_sha256": CODE_SHA256,
    "union_version": U.VERSION,
    "union_sha256": UNION_SHA256,
    "checkpoint_sha256": CHECKPOINT_SHA256,
    "notebook_build": NOTEBOOK_BUILD,
    "lkh_source": LKH_SOURCE,
    "lkh_findtour_sha256": B.LKH_FINDTOUR_SHA256,
    "eax_source": EAX_SOURCE,
    "eax_commit": B.EAX_COMMIT,
    "hardware": HARDWARE,
}
manifest_path = OUTPUT_ROOT / "manifest.json"
if manifest_path.exists():
    saved = json.loads(manifest_path.read_text())
    for key in ["schema", "design", "seeds", "workers", "repo_commit", "code_version", "code_sha256", "union_version",
                "union_sha256", "checkpoint_sha256", "notebook_build", "eax_commit"]:
        assert saved[key] == MANIFEST[key], f"{key} değişti. Ayrı bir EXPERIMENT_NAME kullanın."
    assert saved["protocol"]["sha256"] == PROTOCOL_SHA256, "Protokol değişti. Ayrı bir EXPERIMENT_NAME kullanın."
    assert saved["hardware"]["vcpus_visible"] == HARDWARE["vcpus_visible"], "vCPU sayısı değişti; aynı ortam türünü seçin."
    same_machine_type = (saved["hardware"]["gpu"] == HARDWARE["gpu"]
                         and saved["hardware"]["cpu_model"] == HARDWARE["cpu_model"])
    assert ALLOW_HARDWARE_CHANGE or same_machine_type, \
        (f"Makine türü değişti ({saved['hardware']['gpu']}, {saved['hardware']['cpu_model']} -> {HARDWARE['gpu']}, "
         f"{HARDWARE['cpu_model']}). Aynı türü seçin; bulunamıyorsa ilk hücrede ALLOW_HARDWARE_CHANGE = True yapın "
         "(her örneğin bütün kolları yine tek makinede koşar).")
    if saved["hardware"] != HARDWARE:
        print("UYARI: donanım veya kütüphane sürümleri ilk oturumdan farklı; her örneğin donanımı ayrıca kaydedilir.")
else:
    manifest_path.write_text(json.dumps(MANIFEST, indent=2, ensure_ascii=False))

# Aynı Colab sanal makinesinde defter yeniden çalıştırılırsa bu işaret korunur. Yeni makinede yarım kalan
# örnek baştan çalıştırılır; böylece bir örneğin bütün kolları tek makinede koşar.
session_path = WORK_ROOT / "runtime_session_id.txt"
if session_path.exists():
    SESSION_ID = session_path.read_text().strip()
else:
    SESSION_ID = uuid.uuid4().hex
    session_path.write_text(SESSION_ID)

print("Hazır:", CPU_COUNT, "vCPU |", RAM_GB, "GB RAM |", CPU_MODEL, "| GPU:", GPU_NAME, "| LKH:", LKH_SOURCE,
      "| GA-EAX:", EAX_SOURCE, "| SCIP", SCIP_VERSION, "| kod", B.VERSION, CODE_SHA256[:12], U.VERSION,
      UNION_SHA256[:12], "| sonuçlar:", OUTPUT_ROOT)

In [ ]:
def round_robin(lists):
    out = []
    for i in range(max(len(x) for x in lists)):
        out += [x[i] for x in lists if i < len(x)]
    return out


def make_instances():
    """Yeni sentetik örnekler. Sıra: boyuta göre, her boyutta aileler dönüşümlü (önce 10.000 düğüm)."""
    groups = {}
    for group, n, count, family, gen_seed, budget in SYNTHETIC:
        rng = np.random.default_rng(gen_seed)
        groups[group] = [dict(group=group, family=family, index=i, name=f"u27_{group}_{i:03d}",
                              xy=T.to_int_coords(T.gen_instance(n, family, rng)), ew="EUC_2D", problem_file=None,
                              budget=float(budget), optimum=None) for i in range(count)]
    result = []
    for size in sorted({n for _, n, *_ in SYNTHETIC}):
        result += round_robin([groups[g] for g, n, *_ in SYNTHETIC if n == size])
    assert len({x["name"] for x in result}) == len(result) == sum(len(v) for v in groups.values())
    return result


INSTANCES = make_instances()


def estimated_seconds(inst):
    """Örnek başına: kayıtlı LKH koşusu (bütçe), iki GA-EAX süreci, LKH'nin kendi birleştirmesi ve üç tam çözüm
    süreci (her biri bütçenin kalanı; tam çözüm süreçleri aynı anda koşar) ve küçük ek süreler."""
    n, budget = len(inst["xy"]), inst["budget"]
    return (budget + STOP_PAD_S) + 3.0 * (1.0 - SPLIT) * budget + 0.002 * n + 25.0


by_group = {}
for x in INSTANCES:
    by_group[x["group"]] = by_group.get(x["group"], 0.0) + estimated_seconds(x) / 3600.0
estimate_h = sum(by_group.values())
print(f"{len(INSTANCES)} örnek hazır. Tahmini süre: yaklaşık {estimate_h:.1f} saat ("
      + ", ".join(f"{g}: {h:.1f} sa" for g, h in by_group.items()) + ").")

In [ ]:
POP_PARAMS = {"CANDIDATE_SET_TYPE": "POPMUSIC", "SUBGRADIENT": "NO", "MAX_CANDIDATES": K_LIST}
FRACTIONS = [0.25, 0.5, 0.75, 1.0]
SPLIT_TAG = f"{int(round(100 * SPLIT)):02d}"
# GA-EAX'in iki popülasyonu. "trials": her sürecin en iyi turu + son atılan deneme turları; "best6": yalnızca altı
# sürecin en iyi turları. Birincil kol (dual) ikisini aynı anda iki çekirdekte koşturur ve kısa olan turu alır.
TRIALS_ARM, BEST6_ARM = "trials_" + SPLIT_TAG, "best6_" + SPLIT_TAG
DUAL_ARM, LKHMERGE_ARM = "dual_" + SPLIT_TAG, "lkhmerge_" + SPLIT_TAG
EAX_ARMS = [(TRIALS_ARM, "trials"), (BEST6_ARM, "best6")]
# Birleşim grafında tam çözüm (SCIP): altı en iyi turun birleşimi; aynı birleşim + BetaTour'un en olası yeni kenarları;
# aynı birleşim + aynı sayıda en kısa yeni kenar (kontrol). Üçü de tek çekirdekte, aynı anda koşar.
EXACT_ARM, EXACT_BETA_ARM, EXACT_NEAR_ARM = "exact6_" + SPLIT_TAG, "exact6beta_" + SPLIT_TAG, "exact6near_" + SPLIT_TAG
EXACT_ARMS = [(EXACT_ARM, None), (EXACT_BETA_ARM, "probability"), (EXACT_NEAR_ARM, "length")]
ARM_NAMES = ["lkh6", TRIALS_ARM, BEST6_ARM, DUAL_ARM, LKHMERGE_ARM, EXACT_ARM, EXACT_BETA_ARM, EXACT_NEAR_ARM]
TOUR_ARMS = ["lkh6", TRIALS_ARM, BEST6_ARM, LKHMERGE_ARM, EXACT_ARM, EXACT_BETA_ARM, EXACT_NEAR_ARM]


def write_json_atomic(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, indent=1, ensure_ascii=False, allow_nan=False))
    os.replace(tmp, path)


def write_npz_atomic(path, **arrays):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.stem + ".tmp.npz")
    np.savez_compressed(tmp, **arrays)
    os.replace(tmp, path)


def parallel(func, items):
    with ThreadPoolExecutor(max_workers=W) as pool:
        return list(pool.map(func, items))


def xy_hash(inst):
    return hashlib.sha256(np.ascontiguousarray(inst["xy"]).tobytes()).hexdigest()


def instance_files(inst):
    """Örneğin yerel diskteki klasörü, LKH'nin okuduğu TSPLIB dosyası ve GA-EAX'in okuduğu dosya."""
    folder = WORK_ROOT / "local" / inst["name"]
    folder.mkdir(parents=True, exist_ok=True)
    lkh_problem = inst["problem_file"] or str(folder / "p.tsp")
    if inst["problem_file"] is None and not Path(lkh_problem).exists():
        T.write_tsp(lkh_problem, inst["xy"], name=inst["name"], ew=inst["ew"])
    eax_problem = folder / "p_eax.tsp"
    if not eax_problem.exists():
        assert inst["ew"] == "EUC_2D"
        B.write_eax_problem(str(eax_problem), inst["xy"])
    return folder, lkh_problem, str(eax_problem)


def eax_version(n):
    return "Normal" if n <= EAX_NORMAL_MAX_N else "Large"


def tour_len(inst, tour):
    return int(T.tour_length(inst["xy"], tour, inst["ew"]))


def cut_times(budget):
    """LKH süreçlerinin durdurulduğu an: milisaniye (tam sayı) ve saniye."""
    cut_ms = int(math.floor(1000.0 * SPLIT * budget))
    return cut_ms, cut_ms / 1000.0


def checked_eax_tour(inst, out, best_file, label):
    """GA-EAX'in yazdığı tur: geçerli olmalı ve bildirdiği uzunluk kendi hesabımızla aynı olmalı."""
    n = len(inst["xy"])
    assert out["finished"], f"{label}: GA-EAX sonuç yazmadı (çıkış kodu {out['returncode']}): {out['tail']}"
    tour, value = B.read_eax_best(best_file, n)
    assert B.is_permutation(tour, n), f"{label}: GA-EAX turu geçersiz"
    assert tour_len(inst, tour) == value, f"{label}: GA-EAX'in bildirdiği uzunluk turun uzunluğuna eşit değil"
    return tour, value


def eax_record(out, value, version):
    return dict(version=version, generations=out["generations"], converged=bool(not out["stopped_by_deadline"]),
                stopped_by_deadline=bool(out["stopped_by_deadline"]), wall_s=float(out["wall_s"]),
                first_line_s=out["first_line_s"], last_line_s=out["last_line_s"], last_stage=out["last_stage"],
                curve=out["curve"], final_length=int(value), returncode=out["returncode"])


def make_recorded(inst):
    """Altı kayıt eden LKH süreci (POPMUSIC-5, kesintisiz; önceki koşulardaki pop_full kolunun ayarları). Her
    denemenin turu ve her yeni en iyi tur, duvar saati damgasıyla dosyaya yazılır; her süreç ayrıca aday listelerini
    LKH'nin kendi biçimiyle dosyaya yazar (LKH'nin kendi tur birleştirmesi bu dosyayı okur; arama değişmez, öz sınama
    denetler). Süreçler bütçeden STOP_PAD_S saniye sonra dışarıdan durdurulur; bütün zamanlar altı sürecin
    başlatıldığı andan itibaren duvar saatidir."""
    n, budget = len(inst["xy"]), inst["budget"]
    folder, lkh_problem, _ = instance_files(inst)
    rec_dir = folder / "rec"
    shutil.rmtree(rec_dir, ignore_errors=True)
    rec_dir.mkdir(parents=True)
    start = time.time()

    def one(seed):
        par = rec_dir / f"s{seed}.par"
        B.write_par(par, dict(PROBLEM_FILE=lkh_problem, RUNS=1, TRACE_LEVEL=1, TIME_LIMIT=round(1.25 * budget + 30.0, 1),
                              MAX_TRIALS=100000000, SEED=seed, **POP_PARAMS, CANDIDATE_FILE=rec_dir / f"cand_s{seed}.txt",
                              OUTPUT_TOUR_FILE=rec_dir / f"s{seed}.tour"))
        return B.run_process([LKH_REC, par], rec_dir, env={"BETATOUR_TRIAL_DUMP": rec_dir / f"s{seed}.bin"},
                             timeout=max(1.0, start + budget + STOP_PAD_S - time.time()))

    raw = parallel(one, SEEDS)
    wall = time.time() - start
    start_ms = int(round(start * 1000.0))
    budget_ms = int(math.floor(1000.0 * budget))
    runs, finals = [], []
    for seed, res in zip(SEEDS, raw):
        assert res["timed_out"] or res["returncode"] == 0, f"lkh6 / tohum {seed}: LKH çıkış kodu {res['returncode']}"
        assert (rec_dir / f"cand_s{seed}.txt").exists(), f"lkh6 / tohum {seed}: aday dosyası yazılmadı"
        records = B.read_records(rec_dir / f"s{seed}.bin", n)
        within = records[records["epoch_ms"] <= start_ms + budget_ms]
        best = within[within["kind"] == 1]
        assert len(best), f"lkh6 / tohum {seed}: bütçe içinde tur yok"
        tour = B.tour_of(best[-1])
        assert B.is_permutation(tour, n) and tour_len(inst, tour) == int(best[-1]["cost"]), \
            f"lkh6 / tohum {seed}: kaydedilen en iyi tur geçersiz"
        curve = B.improvements(records, start_ms)
        trial_end = (records["epoch_ms"][records["kind"] == 0] - start_ms) / 1000.0
        assert B.best_before(curve, budget_ms / 1000.0) == int(best[-1]["cost"])
        runs.append(dict(seed=int(seed), improvements=curve, best_at_budget=int(best[-1]["cost"]),
                         best_at_fraction=[B.best_before(curve, math.floor(1000.0 * f * budget) / 1000.0)
                                           for f in FRACTIONS],
                         trials_at_fraction=[int((trial_end <= math.floor(1000.0 * f * budget) / 1000.0).sum())
                                             for f in FRACTIONS],
                         pre_time_s=B.lkh_preprocessing_s(res["stdout"]), first_tour_s=curve[0][0],
                         last_record_s=float((int(records["epoch_ms"][-1]) - start_ms) / 1000.0),
                         stopped_from_outside=bool(res["timed_out"]), returncode=res["returncode"],
                         wall_s=float(res["wall"])))
        finals.append((int(best[-1]["cost"]), tour))
    at_fraction = []
    for j in range(len(FRACTIONS)):
        known = [r["best_at_fraction"][j] for r in runs if r["best_at_fraction"][j] is not None]
        at_fraction.append(min(known) if known else None)
    record = dict(runs=runs, best_at_budget=min(r["best_at_budget"] for r in runs),
                  mean_at_budget=float(np.mean([r["best_at_budget"] for r in runs])), best_at_fraction=at_fraction,
                  mean_trials_at_fraction=[float(np.mean([r["trials_at_fraction"][j] for r in runs]))
                                           for j in range(len(FRACTIONS))],
                  wall_s=float(wall), start_ms=start_ms)
    return record, {"lkh6": min(finals, key=lambda x: x[0])[1]}


def recorded_at_cut(inst, rec):
    """Kayıtlı koşunun dosyaları ve bütçenin SPLIT anındaki en iyi uzunluk (altı sürecin en iyisi)."""
    n, budget = len(inst["xy"]), inst["budget"]
    folder = instance_files(inst)[0]
    records = [B.read_records(folder / "rec" / f"s{seed}.bin", n) for seed in SEEDS]
    assert all(len(r) for r in records), "kayıt dosyaları eksik"
    cut_ms, cut_s = cut_times(budget)
    known = [B.best_before(run["improvements"], cut_s) for run in rec["runs"]]
    assert all(x is not None for x in known), \
        f"bütçenin %{100 * SPLIT:.0f} anında turu olmayan süreç var (ilk turlar: {[r['first_tour_s'] for r in rec['runs']]})"
    return records, cut_ms, cut_s, int(min(known))


def make_dual(inst, rec):
    """Birincil kol. Altı LKH süreci bütçenin SPLIT anında durur; kayıtlı turlarından iki popülasyon kurulur ve iki
    GA-EAX süreci aynı anda (iki çekirdekte) koşar. Her birine kalan süre, kendi popülasyonunu hazırlama süresi
    düşülerek verilir; sürecin kendi duvar saati sınırı vardır. Kolun sonucu iki turdan kısa olanıdır. Bir
    popülasyondaki turların hepsi aynıysa birleştirilecek bir şey yoktur: GA-EAX koşmaz, sonuç o turdur."""
    n, budget = len(inst["xy"]), inst["budget"]
    folder, _, eax_problem = instance_files(inst)
    records, cut_ms, cut_s, best_at_cut = recorded_at_cut(inst, rec)
    version = eax_version(n)
    jobs = []
    for name, kind in EAX_ARMS:
        begin = time.perf_counter()
        tours, costs, info = B.select_population(records, rec["start_ms"] + cut_ms, PER_PROCESS, n,
                                                 trial_tours=(kind == "trials"))
        pop_file = folder / (name + ".pop")
        B.write_population(pop_file, tours, costs)
        prep_s = time.perf_counter() - begin
        for tour, cost in zip(tours, costs):
            assert B.is_permutation(tour, n) and tour_len(inst, tour) == cost, f"{name}: kaydedilen tur geçersiz"
        assert min(costs) == best_at_cut, f"{name}: popülasyonun en iyi turu altı sürecin en iyi turu değil"
        jobs.append(dict(name=name, kind=kind, prep_s=float(prep_s), pop_file=pop_file, pop_size=len(tours),
                         best_tour=tours[int(np.argmin(costs))], per_process=info,
                         population=B.population_stats(tours, n)))

    def one(job):
        allowed = max(0.0, budget - cut_s - job["prep_s"])
        if job["population"]["distinct"] < 2:
            return allowed, None, None
        out, best_file = B.run_eax(EAX[version], eax_problem, str(folder / ("eax_" + job["name"])), "M",
                                   job["pop_size"], kids=EAX_KIDS, population_file=str(job["pop_file"]),
                                   deadline_epoch=time.time() + allowed)
        return allowed, out, best_file

    raw = parallel(one, jobs)
    arms, tours_out = {}, {}
    for job, (allowed, out, best_file) in zip(jobs, raw):
        if out is None:
            tour, at_budget, eax = job["best_tour"], best_at_cut, None
        else:
            tour, value = checked_eax_tour(inst, out, best_file, job["name"])
            at_deadline = B.best_before(out["curve"], allowed)
            at_budget = min(x for x in (best_at_cut, at_deadline) if x is not None)
            eax = eax_record(out, value, version)
        arms[job["name"]] = dict(kind=job["kind"], split=float(SPLIT), cut_s=float(cut_s), prep_s=job["prep_s"],
                                 eax_budget_s=float(allowed), pop_size=job["pop_size"], best_at_cut=best_at_cut,
                                 population=job["population"], per_process=job["per_process"], eax=eax,
                                 skipped=bool(out is None), at_budget=int(at_budget))
        tours_out[job["name"]] = tour
        Path(job["pop_file"]).unlink(missing_ok=True)
        shutil.rmtree(folder / ("eax_" + job["name"]), ignore_errors=True)
    shortest = min(arms[a]["at_budget"] for a, _ in EAX_ARMS)
    record = dict(arms=arms, at_budget=int(shortest), best_at_cut=best_at_cut,
                  source=[a for a, _ in EAX_ARMS if arms[a]["at_budget"] == shortest],
                  eax_processes=int(sum(not arms[a]["skipped"] for a, _ in EAX_ARMS)))
    return record, tours_out


def make_lkhmerge(inst, rec):
    """Kontrol: LKH-3'ün kendi tur birleştirmesi. Altı LKH süreci bütçenin SPLIT anında durur; altı en iyi turları
    MERGE_TOUR_FILE olarak altı yeni LKH sürecine verilir. LKH bu kipte altı turun ortak kenarlarını sabit tutar,
    turların kenarlarını aday listelerine ekler ve kalanını arar (IsCommonEdge.c, AddTourCandidates.c). Her süreç
    altı turun en iyisinden başlar (INITIAL_TOUR_FILE) ve kayıtlı koşudaki aynı sıradaki sürecin POPMUSIC-5 aday
    dosyasını okur (ön işleme yeniden yapılmaz). Süreçler kalan sürenin sonunda dışarıdan durdurulur; yalnızca süre
    içinde kaydedilen turlar sayılır. Altı tur aynıysa birleştirilecek bir şey yoktur: sonuç o turdur."""
    n, budget = len(inst["xy"]), inst["budget"]
    folder, lkh_problem, _ = instance_files(inst)
    rec_dir, mdir = folder / "rec", folder / "lkhmerge"
    records, cut_ms, cut_s, best_at_cut = recorded_at_cut(inst, rec)
    assert all((rec_dir / f"cand_s{seed}.txt").exists() for seed in SEEDS), "aday dosyaları eksik"
    shutil.rmtree(mdir, ignore_errors=True)
    mdir.mkdir(parents=True)
    begin = time.perf_counter()
    tours, costs, _ = B.select_population(records, rec["start_ms"] + cut_ms, 1, n, trial_tours=False)
    assert len(tours) == W and min(costs) == best_at_cut
    files = []
    for i, (tour, cost) in enumerate(zip(tours, costs)):
        assert B.is_permutation(tour, n) and tour_len(inst, tour) == cost, "lkhmerge: kaydedilen tur geçersiz"
        files.append(mdir / f"t{i}.tour")
        T.write_tour(str(files[-1]), tour, name=f"t{i}", length=cost)
    prep_s = time.perf_counter() - begin
    best_index = int(np.argmin(costs))
    common = B.common_edge_keys(tours, n)
    allowed = max(0.0, budget - cut_s - prep_s)
    record = dict(split=float(SPLIT), cut_s=float(cut_s), prep_s=float(prep_s), lkh_budget_s=float(allowed),
                  tours=len(tours), distinct=int(B.distinct_tours(tours, n)), common_edges=int(len(common)),
                  best_at_cut=best_at_cut)
    if record["distinct"] < 2:
        record.update(skipped=True, runs=[], processes_with_tour=0, at_deadline=None, at_budget=best_at_cut,
                      keeps_common_edges=None, wall_s=0.0)
        shutil.rmtree(mdir, ignore_errors=True)
        return record, {LKHMERGE_ARM: tours[best_index]}
    launch = time.time()
    launch_ms, allowed_ms = int(round(launch * 1000.0)), int(math.floor(1000.0 * allowed))

    def one(i):
        seed = MERGE_SEEDS[i]
        par = mdir / f"m{seed}.par"
        B.write_par(par, dict(PROBLEM_FILE=lkh_problem, RUNS=1, TRACE_LEVEL=2, TIME_LIMIT=round(1.25 * budget + 30.0, 1),
                              MAX_TRIALS=100000000, SEED=seed, **POP_PARAMS,
                              CANDIDATE_FILE=rec_dir / f"cand_s{SEEDS[i]}.txt", INITIAL_TOUR_FILE=files[best_index],
                              OUTPUT_TOUR_FILE=mdir / f"m{seed}.tour"), merge_tour_files=files)
        return B.run_process([LKH_REC, par], mdir,
                             env={"BETATOUR_TRIAL_DUMP": mdir / f"m{seed}.bin", "BETATOUR_BEST_ONLY": "1"},
                             timeout=max(1.0, launch + allowed + STOP_PAD_S - time.time()))

    raw = parallel(one, range(W))
    runs, best = [], None
    for i, res in enumerate(raw):
        seed = MERGE_SEEDS[i]
        found = B.read_records(mdir / f"m{seed}.bin", n)
        assert not len(found) or set(found["kind"].tolist()) == {1}, "lkhmerge: beklenmeyen kayıt türü"
        within = found[found["epoch_ms"] <= launch_ms + allowed_ms]
        curve = B.improvements(found, launch_ms)
        value, keeps = None, None
        if len(within):
            value, tour = int(within[-1]["cost"]), B.tour_of(within[-1])
            assert B.is_permutation(tour, n) and tour_len(inst, tour) == value, \
                f"lkhmerge / tohum {seed}: kaydedilen tur geçersiz"
            keeps = bool(np.isin(common, B.edge_keys(tour, n), assume_unique=True).all())
            if best is None or value < best[0]:
                best = (value, tour)
        failed = not (res["timed_out"] or res["returncode"] == 0) or value is None
        runs.append(dict(seed=int(seed), candidates_of_seed=int(SEEDS[i]), improvements=curve, best_in_time=value,
                         records=int(len(found)), trials=int(len(B.lkh_trial_costs(res["stdout"]))),
                         first_record_s=curve[0][0] if curve else None, keeps_common_edges=keeps,
                         stopped_from_outside=bool(res["timed_out"]), returncode=res["returncode"],
                         wall_s=float(res["wall"]), tail=res["stdout"][-300:] if failed else None))
    assert best is not None, "lkhmerge: hiçbir süreç süresi içinde tur kaydetmedi: " + str(runs[0]["tail"])
    at_budget = min(best_at_cut, best[0])
    record.update(skipped=False, runs=runs, processes_with_tour=int(sum(r["best_in_time"] is not None for r in runs)),
                  at_deadline=int(best[0]), at_budget=int(at_budget),
                  keeps_common_edges=bool(all(r["keeps_common_edges"] for r in runs if r["best_in_time"] is not None)),
                  wall_s=float(time.time() - launch))
    shutil.rmtree(mdir, ignore_errors=True)
    return record, {LKHMERGE_ARM: best[1] if best[0] <= best_at_cut else tours[best_index]}



def first_trial_tours(records, n):
    """Her sürecin ilk denemesinin turu (modelin okuduğu topluluk; run v4-v8 ve v21-v23'teki gibi M = 6)."""
    tours = []
    for r in records:
        first = r[(r["kind"] == 0) & (r["trial"] == 1)]
        assert len(first) == 1, "ilk deneme turu kayıtlı değil"
        tours.append(B.tour_of(first[0]))
    return tours


def model_graph(xy, succ):
    """BetaTour'un grafiği (16 en yakın komşu + modelin topluluk kenarları), model çalıştırılmadan; run v21'deki gibi."""
    xyf = np.asarray(xy, dtype=np.float64)
    span = float((xyf.max(0) - xyf.min(0)).max()) or 1.0
    xyn = (xyf - xyf.min(0)) / span
    nb, _ = M.knn_np(xyn, K_GRAPH)
    extra = int(MODEL.cfg.get("graph_extra", 0))
    if extra:
        nb, _, _ = M.extend_graph(xyn, nb, succ, extra)
    return np.asarray(nb, dtype=np.int64)


def make_exact(inst, rec):
    """Üç tam çözüm kolu. Altı LKH süreci bütçenin SPLIT anında durur; altı en iyi turun birleşim grafı (ve iki kolda
    ek kenarlar) SCIP ile tam çözülür (unionx_v27.solve_exact; başlangıç çözümü altı turun en iyisi). Her kol kendi
    sürecinde, tek çekirdekte, kendi duvar saati sınırıyla koşar; üçü aynı anda başlar. Bir kolun süresi bütçeden
    kesme anı ve kendi hazırlığı (birleşim; model kolunda BetaTour çıkarımı; kontrol kolunda modelin grafiği)
    düşülerek verilir. Ek kenar sayısı EXTRA_FRACTION x n; adaylar modelin grafiğindeki, birleşimde olmayan kenarlardır."""
    n, budget = len(inst["xy"]), inst["budget"]
    folder = instance_files(inst)[0]
    xdir = folder / "exact"
    shutil.rmtree(xdir, ignore_errors=True)
    xdir.mkdir(parents=True)
    records, cut_ms, cut_s, best_at_cut = recorded_at_cut(inst, rec)
    begin = time.perf_counter()
    tours, costs, _ = B.select_population(records, rec["start_ms"] + cut_ms, 1, n, trial_tours=False)
    assert len(tours) == W and min(costs) == best_at_cut
    for tour, cost in zip(tours, costs):
        assert B.is_permutation(tour, n) and tour_len(inst, tour) == cost, "exact: kaydedilen tur geçersiz"
    union = U.union_keys(tours, n)
    start = tours[int(np.argmin(costs))]
    common_s = time.perf_counter() - begin
    first = first_trial_tours(records, n)
    succ = np.stack([T.tour_succ_pred(t)[0] for t in first], axis=1).astype(np.int32)
    k_extra = int(round(EXTRA_FRACTION * n))
    jobs, graphs = [], {}
    for name, by in EXACT_ARMS:
        begin = time.perf_counter()
        model_info = None
        if by is None:
            keys, extra = union, np.zeros(0, dtype=np.int64)
        elif by == "probability":
            first_costs = np.array([T.tour_length(inst["xy"], t, inst["ew"]) for t in first], dtype=np.float64)
            prediction = M.predict_tourfold(MODEL, inst["xy"], K_GRAPH, DEVICE, ens_succ=succ, ens_cost=first_costs)
            graphs[by] = np.asarray(prediction["NB"], dtype=np.int64)
            table = U.probability_table(prediction["prob"], prediction["NB"])
            extra = U.extra_edges(table, union, k_extra, "probability")
            chosen_p = table[1][np.searchsorted(table[0], extra)]      # ek kenarlar modelin grafiğindedir
            keys = np.union1d(union, extra)
            model_info = dict(t_prep_s=float(prediction["t_prep"]), t_model_s=float(prediction["t_model"]),
                              graph_edges=int(len(table[0])), candidates=int((~np.isin(table[0], union)).sum()),
                              prob_min=float(chosen_p.min()) if len(chosen_p) else None,
                              prob_mean=float(chosen_p.mean()) if len(chosen_p) else None)
        else:
            nb = model_graph(inst["xy"], succ)
            graphs[by] = nb
            table = U.probability_table(np.zeros(nb.shape), nb)
            extra = U.extra_edges(table, union, k_extra, "length", xy=inst["xy"], ew=inst["ew"])
            keys = np.union1d(union, extra)
        in_file, out_file = xdir / f"{name}_in.npz", xdir / f"{name}_out.npz"
        np.savez(in_file, xy=inst["xy"], keys=keys, start=start, deadline=np.float64(0.0), ew=np.array(inst["ew"]))
        prep_s = common_s + (time.perf_counter() - begin)
        jobs.append(dict(name=name, by=by, keys=keys, extra=extra, prep_s=float(prep_s), model=model_info,
                         in_file=in_file, out_file=out_file))
    assert np.array_equal(graphs["probability"], graphs["length"]), "model grafiği ile kontrol grafiği farklı"
    launch = time.time()

    def one(job):
        allowed = max(0.0, budget - cut_s - job["prep_s"])
        deadline = launch + allowed
        with np.load(job["in_file"], allow_pickle=False) as z:
            data = {k: z[k] for k in z.files}
        data["deadline"] = np.float64(deadline)
        np.savez(job["in_file"], **data)
        res = B.run_process([sys.executable, UNION_PATH, "solve", job["in_file"], job["out_file"]], xdir,
                            timeout=allowed + EXACT_PAD_S)
        return allowed, deadline, res

    raw = parallel(one, jobs)
    arms, tours_out = {}, {}
    for job, (allowed, deadline, res) in zip(jobs, raw):
        assert res["returncode"] == 0 and Path(job["out_file"]).exists(), \
            f"{job['name']}: tam çözüm süreci başarısız (zaman aşımı: {res['timed_out']}): {res['stdout'][-1500:]}"
        with np.load(job["out_file"], allow_pickle=False) as z:
            tour, info = z["tour"].astype(np.int64), json.loads(str(z["info"]))
        keys = job["keys"]
        assert B.is_permutation(tour, n) and np.isin(U.tour_keys(tour), keys).all(), f"{job['name']}: tur geçersiz"
        length = tour_len(inst, tour)
        assert length == info["length"] and length <= best_at_cut, f"{job['name']}: uzunluk tutarsız"
        found_epoch = (info["solve_start_epoch"] + info["time_to_best_s"]
                       if info.get("solve_start_epoch") is not None and info.get("time_to_best_s") is not None else None)
        late = bool(found_epoch is not None and found_epoch > deadline + 1.0 and length < best_at_cut)
        tour_set = U.tour_keys(tour)
        arms[job["name"]] = dict(
            by=job["by"], split=float(SPLIT), cut_s=float(cut_s), prep_s=job["prep_s"], exact_budget_s=float(allowed),
            best_at_cut=best_at_cut, union_edges=int(len(union)), extra_edges=int(len(job["extra"])),
            extra_used=int(np.isin(job["extra"], tour_set).sum()), outside_union=int((~np.isin(tour_set, union)).sum()),
            model=job["model"], solver=info, finished_after_deadline_s=round(float(info["finished_epoch"] - deadline), 3),
            best_found_late=late, at_budget=int(best_at_cut if late else length))
        tours_out[job["name"]] = tour if not late else start
    record = dict(arms=arms, best_at_cut=best_at_cut, union_edges=int(len(union)), k_extra=k_extra,
                  pairwise=B.population_stats(tours, n), wall_s=float(time.time() - launch))
    tours_out["cut_tours"] = np.stack(tours).astype(np.int32)
    shutil.rmtree(xdir, ignore_errors=True)
    return record, tours_out


STEP_ORDERS = [("dual", "lkhmerge", "exact"), ("lkhmerge", "exact", "dual"), ("exact", "dual", "lkhmerge")]


def run_one_instance(inst, ordinal):
    name, n = inst["name"], len(inst["xy"])
    final_path = OUTPUT_ROOT / "instances" / (name + ".json")
    stage_dir = OUTPUT_ROOT / "stages" / name
    local = WORK_ROOT / "local" / name
    session_marker = stage_dir / "runtime_session_id.txt"
    if stage_dir.exists() and any(stage_dir.iterdir()):
        old_id = session_marker.read_text().strip() if session_marker.exists() else None
        if old_id != SESSION_ID:
            print("Yeni Colab makinesi: yarım kalan örnek baştan çalıştırılıyor:", name, flush=True)
            shutil.rmtree(stage_dir)
            shutil.rmtree(local, ignore_errors=True)
    stage_dir.mkdir(parents=True, exist_ok=True)
    session_marker.write_text(SESSION_ID)
    stage = {key: (stage_dir / f"{key}.json", stage_dir / f"tours_{key}.npz")
             for key in ("lkh6", "dual", "lkhmerge", "exact")}
    # Türev kolların sırası örnek numarasına göre döner (üç sıra); hepsi aynı kayıtlı koşudan çıkar.
    order = STEP_ORDERS[inst["index"] % len(STEP_ORDERS)]

    def save(key, record, tours):
        write_npz_atomic(stage[key][1], **{k: np.asarray(v).astype(np.int32) for k, v in tours.items()})
        write_json_atomic(stage[key][0], record)

    derived = {"dual": (make_dual, "iki GA-EAX süreci (" + ", ".join(a for a, _ in EAX_ARMS) + ")"),
               "lkhmerge": (make_lkhmerge, "LKH'nin kendi tur birleştirmesi"),
               "exact": (make_exact, "birleşim grafında üç tam çözüm (" + ", ".join(a for a, _ in EXACT_ARMS) + ")")}
    if not all(stage[key][0].exists() for key in stage):
        files = all((local / "rec" / f"s{seed}.bin").exists() and (local / "rec" / f"cand_s{seed}.txt").exists()
                    for seed in SEEDS)
        if not (stage["lkh6"][0].exists() and files):
            for key in derived:                # türev kolların hepsi aynı kayıtlı koşudan çıkmalıdır
                for path in stage[key]:
                    path.unlink(missing_ok=True)
            print(name, "/ altı kayıtlı LKH süreci", flush=True)
            save("lkh6", *make_recorded(inst))
        rec = json.loads(stage["lkh6"][0].read_text())
        for key in order:
            if not stage[key][0].exists():
                func, label = derived[key]
                print(name, "/", label, flush=True)
                save(key, *func(inst, rec))
        shutil.rmtree(local / "rec", ignore_errors=True)
    rec, dual, merge, exact = (json.loads(stage[key][0].read_text()) for key in ("lkh6", "dual", "lkhmerge", "exact"))
    results = {"lkh6": int(rec["best_at_budget"]), DUAL_ARM: int(dual["at_budget"]), LKHMERGE_ARM: int(merge["at_budget"])}
    results.update({arm: int(dual["arms"][arm]["at_budget"]) for arm, _ in EAX_ARMS})
    results.update({arm: int(exact["arms"][arm]["at_budget"]) for arm, _ in EXACT_ARMS})
    assert set(results) == set(ARM_NAMES)
    assert results[DUAL_ARM] == min(results[arm] for arm, _ in EAX_ARMS)
    assert dual["best_at_cut"] == merge["best_at_cut"] == exact["best_at_cut"] >= rec["best_at_budget"]
    tours = {}
    for key in stage:
        with np.load(stage[key][1], allow_pickle=False) as z:
            tours.update({k: z[k] for k in z.files})
    assert set(tours) == set(TOUR_ARMS) | {"cut_tours"}
    final_lengths = {arm: tour_len(inst, tours[arm].astype(np.int64)) for arm in TOUR_ARMS}
    assert all(B.is_permutation(tours[arm].astype(np.int64), n) for arm in TOUR_ARMS), "kaydedilen tur geçersiz"
    assert all(final_lengths[arm] <= results[arm] for arm in TOUR_ARMS), "kaydedilen tur, kaydedilen sonuçtan uzun"
    union = U.union_keys(tours["cut_tours"].astype(np.int64), n)
    outside = {arm: int((~np.isin(U.tour_keys(tours[arm].astype(np.int64)), union)).sum()) for arm in TOUR_ARMS}
    write_npz_atomic(OUTPUT_ROOT / "tours" / (name + ".npz"), **tours)
    data = dict(schema=1, complete=True, name=name, group=inst["group"], family=inst["family"],
                index=int(inst["index"]), ordinal=int(ordinal), n=n, budget_s=float(inst["budget"]), ew=inst["ew"],
                xy_sha256=xy_hash(inst), seeds=SEEDS, merge_seeds=MERGE_SEEDS, fractions=FRACTIONS, split=float(SPLIT),
                order=list(order), hardware=HARDWARE, session_id=SESSION_ID, code_sha256=CODE_SHA256,
                union_sha256=UNION_SHA256, results=results, final_tour_lengths=final_lengths,
                edges_outside_union6=outside, lkh6=rec, dual=dual, lkhmerge=merge, exact=exact,
                finished_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()))
    write_json_atomic(final_path, data)
    shutil.rmtree(local, ignore_errors=True)
    print("TAMAM", name, "|", ", ".join(f"{arm}={results[arm]}" for arm in ARM_NAMES), flush=True)
    return data

In [ ]:
# Öz sınama (ölçüme dahil değildir). Bu makinede ve derlenen ikililerle:
#   1) Kayıt eden LKH (aday dosyasını da yazarken), değiştirilmemiş LKH ile aynı aramayı yapıyor mu? Sabit deneme
#      sayılı (belirlenimci) iki koşu her denemede aynı maliyeti vermelidir.
#   2) Kaydedilen her tur geçerli mi ve uzunluğu kaydedilen maliyete eşit mi? Son "en iyi tur" kaydı, LKH'nin
#      kendi yazdığı tur mu?
#   3) GA-EAX kaydedilen turlardan başlayınca geçerli bir tur veriyor mu, bildirdiği uzunluk kendi hesabımızla aynı
#      mı, en iyi girdi turundan uzun olmayan bir tur mu? İki sürüm de (Normal, Large) denenir.
#   4) GA-EAX duvar saati sınırında en iyi turuyla duruyor mu?
#   5) En yakın şehir listelerini ızgarayla bulan GA-EAX, bu yama olmadan derlenen GA-EAX ile aynı turu, aynı kuşak
#      sayısında veriyor mu?
#   6) LKH'nin kendi tur birleştirmesi (kontrol kolu) beklendiği gibi çalışıyor mu? Aday dosyası okunuyor mu (LKH
#      okuyamadığı dosyayı yeniden yazar), yalnızca en iyi turları kaydeden kip aramayı değiştirmiyor mu, sonuç turu
#      geçerli mi, başlangıç turundan uzun değil mi ve birleştirilen turların ortak kenarlarının hepsini taşıyor mu?
#   7) Birleşim grafında tam çözüm: küçük rastgele graflarda sayımla aynı en kısa turu mu veriyor; kayıtlı altı
#      deneme turunun birleşiminde kanıtlanmış en kısa, geçerli, yalnızca birleşimin kenarlarını kullanan ve en iyi
#      girdi turundan uzun olmayan bir tur mu veriyor; sabitlenen kenarlar her girdi turunda var, silinen kenarlar
#      hiçbirinde yok mu; komut satırından (ayrı süreç, duvar saati sınırıyla) aynı sonucu mu veriyor?
#   8) BetaTour çıkarımı ve ek kenarlar: modelin grafiği, modeli çalıştırmadan kurulan grafikle aynı mı; seçilen ek
#      kenarlar birleşimde yok, modelin grafiğinde var ve istenen sayıda mı?
def selfcheck(n=2000, trials=25, merge_trials=40):
    xy = T.to_int_coords(T.gen_instance(n, "clustered", np.random.default_rng(10093)))
    folder = WORK_ROOT / "selfcheck"
    shutil.rmtree(folder, ignore_errors=True)
    folder.mkdir(parents=True)
    problem, eax_problem = folder / "p.tsp", folder / "p_eax.tsp"
    T.write_tsp(str(problem), xy, name="selfcheck", ew="EUC_2D")
    B.write_eax_problem(str(eax_problem), xy)

    def lkh(binary, tag, extra=None, env=None, merge=(), max_trials=trials):
        par = folder / (tag + ".par")
        B.write_par(par, dict(PROBLEM_FILE=problem, RUNS=1, TRACE_LEVEL=2, MAX_TRIALS=max_trials, SEED=SEEDS[0],
                              **POP_PARAMS, **(extra or {}), OUTPUT_TOUR_FILE=folder / (tag + ".tour")),
                    merge_tour_files=merge)
        res = B.run_process([binary, par], folder, env=env, timeout=900)
        assert res["returncode"] == 0, res["stdout"][-500:]
        return res

    cand = folder / "cand.txt"
    plain = lkh(LKH, "plain")
    recorded = lkh(LKH_REC, "rec", extra={"CANDIDATE_FILE": cand}, env={"BETATOUR_TRIAL_DUMP": folder / "rec.bin"})
    costs_plain, costs_rec = B.lkh_trial_costs(plain["stdout"]), B.lkh_trial_costs(recorded["stdout"])
    records = B.read_records(folder / "rec.bin", n)
    trial_rec, best_rec = records[records["kind"] == 0], records[records["kind"] == 1]
    valid = all(B.is_permutation(B.tour_of(r), n) and int(T.tour_length(xy, B.tour_of(r))) == int(r["cost"])
                for r in records)
    own = T.read_tour(str(folder / "rec.tour"))
    tours, costs, _ = B.select_population([records], int(records["epoch_ms"][-1]), 12, n)
    B.write_population(folder / "pop", tours, costs)
    eax = {}
    for version in ("Normal", "Large"):
        out, best_file = B.run_eax(EAX[version], str(eax_problem), str(folder / ("eax_" + version)), "S", len(tours),
                                   kids=EAX_KIDS, population_file=str(folder / "pop"), deadline_epoch=time.time() + 120)
        tour, value = B.read_eax_best(best_file, n)
        ref, ref_file = B.run_eax(EAX_PLAIN[version], str(eax_problem), str(folder / ("plain_" + version)), "S",
                                  len(tours), kids=EAX_KIDS, population_file=str(folder / "pop"),
                                  deadline_epoch=time.time() + 120)
        ref_tour, ref_value = B.read_eax_best(ref_file, n)
        eax[version] = dict(finished=bool(out["finished"]), valid=bool(B.is_permutation(tour, n)),
                            length_reported=int(value), length_recomputed=int(T.tour_length(xy, tour)),
                            generations=out["generations"], wall_s=float(out["wall_s"]),
                            setup_s=out["first_line_s"], setup_without_grid_s=ref["first_line_s"],
                            same_as_without_grid=bool(ref_value == value and ref["generations"] == out["generations"]
                                                      and np.array_equal(ref_tour, tour)))
    # Sınır şimdiki an: GA-EAX popülasyonunu kurar, ilk denetimde durur ve o andaki en iyi turunu yazar.
    out, best_file = B.run_eax(EAX["Normal"], str(eax_problem), str(folder / "eax_deadline"), "D", EAX_SCRATCH_POP,
                               kids=EAX_KIDS, deadline_epoch=time.time())
    tour, value = B.read_eax_best(best_file, n)
    deadline = dict(finished=bool(out["finished"]), stopped_by_deadline=bool(out["stopped_by_deadline"]),
                    valid=bool(B.is_permutation(tour, n) and int(T.tour_length(xy, tour)) == int(value)),
                    wall_s=float(out["wall_s"]))

    # LKH'nin kendi tur birleştirmesi: kayıtlı koşunun son farklı deneme turlarından en çok altısı birleştirilir.
    chosen, seen = [], set()
    for r in trial_rec[::-1]:
        key = B.edge_keys(B.tour_of(r), n).tobytes()
        if key not in seen:
            seen.add(key)
            chosen.append((int(r["cost"]), B.tour_of(r)))
        if len(chosen) == 6:
            break
    assert len(chosen) >= 2, "öz sınama: birleştirilecek iki farklı deneme turu yok"
    chosen.sort(key=lambda x: x[0])
    files = []
    for i, (cost, tour) in enumerate(chosen):
        files.append(folder / f"t{i}.tour")
        T.write_tour(str(files[-1]), tour, name=f"t{i}", length=cost)
    common = B.common_edge_keys([t for _, t in chosen], n)
    before = (cand.stat().st_mtime_ns, hashlib.sha256(cand.read_bytes()).hexdigest())
    extra = {"CANDIDATE_FILE": cand, "INITIAL_TOUR_FILE": files[0]}
    merge_plain = lkh(LKH, "merge_plain", extra=extra, merge=files, max_trials=merge_trials)
    merge_rec = lkh(LKH_REC, "merge_rec", extra=extra, merge=files, max_trials=merge_trials,
                    env={"BETATOUR_TRIAL_DUMP": folder / "merge.bin", "BETATOUR_BEST_ONLY": "1"})
    after = (cand.stat().st_mtime_ns, hashlib.sha256(cand.read_bytes()).hexdigest())
    merge_costs = B.lkh_trial_costs(merge_rec["stdout"])
    merged = B.read_records(folder / "merge.bin", n)
    final = T.read_tour(str(folder / "merge_rec.tour"))
    final_length = int(T.tour_length(xy, final))
    merge = dict(
        tours=int(len(chosen)), common_edges=int(len(common)), trials=int(merge_trials),
        same_search=bool(len(merge_costs) == merge_trials and merge_costs == B.lkh_trial_costs(merge_plain["stdout"])),
        best_only=bool(len(merged) >= 1 and set(merged["kind"].tolist()) == {1}),
        candidate_file_read=bool(before == after),
        valid=bool(B.is_permutation(final, n)), best_input=int(chosen[0][0]), final_length=final_length,
        not_longer=bool(final_length <= chosen[0][0]),
        keeps_common_edges=bool(np.isin(common, B.edge_keys(final, n), assume_unique=True).all()),
        last_record_is_lkh_tour=bool(len(merged) >= 1 and int(merged[-1]["cost"]) == final_length
                                     and np.array_equal(B.edge_keys(B.tour_of(merged[-1]), n), B.edge_keys(final, n))),
        wall_s=float(merge_rec["wall"]), pre_time_s=B.lkh_preprocessing_s(merge_rec["stdout"]))
    result = dict(
        nodes=n, trials=trials,
        same_search=bool(len(costs_plain) == trials and costs_plain == costs_rec),
        candidate_file_written=bool(cand.exists() and cand.stat().st_size > 0),
        trial_tours_recorded=int(len(trial_rec)), best_tours_recorded=int(len(best_rec)),
        recorded_tours_valid=bool(valid and list(trial_rec["trial"]) == list(range(1, trials + 1))),
        last_best_is_lkh_tour=bool(np.array_equal(B.edge_keys(own, n), B.edge_keys(B.tour_of(best_rec[-1]), n))
                                   and int(best_rec[-1]["cost"]) == min(c for _, c in costs_rec)),
        wall_plain_s=float(plain["wall"]), wall_recording_s=float(recorded["wall"]),
        best_input=int(min(costs)), eax=eax, deadline=deadline, merge=merge,
        union=union_check(folder, xy, [t for _, t in chosen]))
    shutil.rmtree(folder, ignore_errors=True)
    return result


def union_check(folder, xy, tours):
    n = len(xy)
    rng = np.random.default_rng(2727)
    small = []
    for _ in range(12):
        m = int(rng.integers(6, 10))
        sxy = rng.integers(0, 1000, size=(m, 2))
        stours = [rng.permutation(m) for _ in range(int(rng.integers(1, 4)))]
        skeys = U.union_keys(stours, m)
        stour, sinfo = U.solve_exact(sxy, skeys, min(stours, key=lambda t: U.tour_length(sxy, t)), 30.0)
        small.append(bool(sinfo["optimal"] and sinfo["length"] == U.brute_force(sxy, skeys)[0]))
    keys = U.union_keys(tours, n)
    lengths = [int(T.tour_length(xy, t)) for t in tours]
    start = tours[int(np.argmin(lengths))]
    red = U.reduce_graph(n, keys)
    consistent = True
    for t in tours:
        mask = np.isin(keys, U.tour_keys(t))
        consistent &= bool(np.all(mask[red["status"] == 1]) and not np.any(mask[red["status"] == 2]))
    tour, info = U.solve_exact(xy, keys, start, 120.0)
    src, dst = folder / "union_in.npz", folder / "union_out.npz"
    np.savez(src, xy=xy, keys=keys, start=start, deadline=np.float64(time.time() + 120.0), ew=np.array("EUC_2D"))
    res = B.run_process([sys.executable, UNION_PATH, "solve", src, dst], folder, timeout=300)
    with np.load(dst, allow_pickle=False) as z:
        cli_tour, cli_info = z["tour"].astype(np.int64), json.loads(str(z["info"]))
    succ = np.stack([T.tour_succ_pred(t)[0] for t in tours], axis=1).astype(np.int32)
    prediction = M.predict_tourfold(MODEL, xy, K_GRAPH, DEVICE, ens_succ=succ,
                                    ens_cost=np.array(lengths, dtype=np.float64))
    table = U.probability_table(prediction["prob"], prediction["NB"])
    k = int(round(EXTRA_FRACTION * n))
    top = U.extra_edges(table, keys, k, "probability")
    near = U.extra_edges(table, keys, k, "length", xy=xy)
    return dict(
        small_graphs_equal_enumeration=bool(all(small)), tours=len(tours), union_edges=int(len(keys)),
        fixed=info["fixed"], removed=info["removed"], core=info["core"], reduction_consistent=bool(consistent),
        optimal=bool(info["optimal"]), valid=bool(B.is_permutation(tour, n) and np.isin(U.tour_keys(tour), keys).all()),
        best_input=int(min(lengths)), length=int(info["length"]), not_longer=bool(info["length"] <= min(lengths)),
        length_recomputed=bool(int(T.tour_length(xy, tour)) == info["length"]), solve_s=info["solve_s"],
        cli_same=bool(res["returncode"] == 0 and cli_info["length"] == info["length"]
                      and int(T.tour_length(xy, cli_tour)) == cli_info["length"]),
        graph_same=bool(np.array_equal(model_graph(xy, succ), np.asarray(prediction["NB"], dtype=np.int64))),
        extra_ok=bool(len(top) == len(near) == k and not np.isin(top, keys).any() and not np.isin(near, keys).any()
                      and np.isin(top, table[0]).all() and np.isin(near, table[0]).all()),
        model_s=float(prediction["t_total"]), scip=SCIP_VERSION)


SELFCHECK = selfcheck(n=600, trials=12, merge_trials=20) if TEST_MODE else selfcheck()
SELFCHECK["session_id"] = SESSION_ID
write_json_atomic(OUTPUT_ROOT / "summary" / "selfcheck.json", SELFCHECK)
assert SELFCHECK["same_search"], "Kayıt eden LKH, değiştirilmemiş LKH ile aynı aramayı yapmadı."
assert SELFCHECK["candidate_file_written"], "Kayıt eden LKH aday dosyasını yazmadı."
assert SELFCHECK["recorded_tours_valid"] and SELFCHECK["last_best_is_lkh_tour"], "Kaydedilen turlar beklenen gibi değil."
for version, item in SELFCHECK["eax"].items():
    assert item["finished"] and item["valid"] and item["length_reported"] == item["length_recomputed"], \
        f"GA-EAX ({version}) geçerli ve uzunluğu doğru bir tur vermedi."
    assert item["length_reported"] <= SELFCHECK["best_input"], f"GA-EAX ({version}) en iyi girdi turundan uzun tur verdi."
    assert item["same_as_without_grid"], f"GA-EAX ({version}): ızgaralı komşu araması yamasız derlemeyle aynı sonucu vermedi."
assert SELFCHECK["deadline"]["finished"] and SELFCHECK["deadline"]["stopped_by_deadline"] and SELFCHECK["deadline"]["valid"], \
    "GA-EAX duvar saati sınırında en iyi turuyla durmadı."
MG = SELFCHECK["merge"]
assert MG["same_search"] and MG["best_only"], "LKH'nin tur birleştirmesi: kayıt eden kopya aramayı değiştirdi ya da deneme turlarını da yazdı."
assert MG["candidate_file_read"], "LKH'nin tur birleştirmesi aday dosyasını okumadı (dosya yeniden yazıldı)."
assert MG["valid"] and MG["not_longer"] and MG["last_record_is_lkh_tour"], "LKH'nin tur birleştirmesi beklenen turu vermedi."
assert MG["keeps_common_edges"], "LKH'nin tur birleştirmesi ortak kenarları korumadı."
X = SELFCHECK["union"]
assert X["small_graphs_equal_enumeration"], "Tam çözüm küçük graflarda sayımla aynı en kısa turu vermedi."
assert X["reduction_consistent"], "İndirgeme bir girdi turuyla çelişti (sabit kenar eksik ya da silinen kenar var)."
assert X["optimal"] and X["valid"] and X["not_longer"] and X["length_recomputed"], "Birleşim grafında tam çözüm beklenen turu vermedi."
assert X["cli_same"], "Komut satırından çalışan tam çözüm aynı sonucu vermedi."
assert X["graph_same"] and X["extra_ok"], "BetaTour grafiği ya da ek kenarlar beklenen gibi değil."
print("Öz sınama tamam. Kayıt eden LKH aynı aramayı yaptı (" + str(SELFCHECK["trials"]) + " deneme; süre "
      f"{SELFCHECK['wall_plain_s']:.1f} s / kayıtla {SELFCHECK['wall_recording_s']:.1f} s). GA-EAX: "
      + ", ".join(f"{v} {x['length_reported']} ({x['generations']} kuşak)" for v, x in SELFCHECK["eax"].items())
      + f"; en iyi girdi turu {SELFCHECK['best_input']}. LKH'nin tur birleştirmesi: {MG['tours']} tur, "
      f"{MG['common_edges']} ortak kenar korundu, {MG['best_input']} -> {MG['final_length']} ({MG['trials']} deneme). "
      f"Birleşimde tam çözüm: {X['tours']} tur, {X['union_edges']} kenar, çekirdek {X['core']} düğüm, "
      f"{X['best_input']} -> {X['length']} ({X['solve_s']:.1f} s, SCIP {X['scip']}).")

In [ ]:
def save_progress(last_instance=None, state="running"):
    finished = sorted(p.stem for p in (OUTPUT_ROOT / "instances").glob("*.json"))
    write_json_atomic(OUTPUT_ROOT / "progress.json",
                      dict(state=state, completed=len(finished), total=len(INSTANCES), finished=finished,
                           last_instance=last_instance, updated_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())))


save_progress(state="running")
session_start = time.time()
newly_finished = 0
for ordinal, inst in enumerate(INSTANCES):
    final_file = OUTPUT_ROOT / "instances" / (inst["name"] + ".json")
    if final_file.exists():
        record = json.loads(final_file.read_text())
        assert record.get("complete") and record.get("xy_sha256") == xy_hash(inst) \
            and set(record["results"]) == set(ARM_NAMES), f"{inst['name']}: eski kayıt bu kurulumla uyuşmuyor"
        print("ATLA (tamam):", inst["name"], flush=True)
        continue
    if MAX_NEW_INSTANCES is not None and newly_finished >= MAX_NEW_INSTANCES:
        print("Bu oturum için MAX_NEW_INSTANCES sınırına ulaşıldı.")
        break
    try:
        run_one_instance(inst, ordinal)
        newly_finished += 1
        save_progress(last_instance=inst["name"])
        done_now = len(list((OUTPUT_ROOT / "instances").glob("*.json")))
        left = sum(estimated_seconds(x) for x in INSTANCES
                   if not (OUTPUT_ROOT / "instances" / (x["name"] + ".json")).exists())
        print(f"  {done_now}/{len(INSTANCES)} örnek bitti; bu oturumda geçen süre "
              f"{(time.time() - session_start) / 60:.0f} dk, kalan yaklaşık {left / 3600:.1f} saat.", flush=True)
    except Exception as exc:
        failure = dict(instance=inst["name"], error=repr(exc), traceback=traceback.format_exc(),
                       utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()))
        write_json_atomic(OUTPUT_ROOT / "errors" / (inst["name"] + ".json"), failure)
        save_progress(last_instance=inst["name"], state="error")
        raise

done = len(list((OUTPUT_ROOT / "instances").glob("*.json")))
save_progress(state="complete" if done == len(INSTANCES) else "partial")
print(f"Bu oturumda yeni {newly_finished}, toplam {done}/{len(INSTANCES)} örnek tamamlandı. "
      f"Süre: {(time.time() - session_start) / 3600:.2f} saat. Kayıt: {OUTPUT_ROOT}")

In [ ]:
# Okumalar (karar kuralı, PROTOCOL_v27.md) ve betimsel karşılaştırmalar. Yüzde fark = 100 x (ilk - ikinci) / ikinci;
# negatif: ilk kol daha kısa. Okumalar 12 örnekte (10.000 düğüm); 20.000 düğüm yalnızca betimseldir. Pilotta
# düzeltme yoktur.
READINGS = [
    ("R1", EXACT_ARM, BEST6_ARM, "Aynı altı tur: birleşimde tam çözüm / GA-EAX"),
    ("R2", EXACT_ARM, DUAL_ARM, "Birleşimde tam çözüm / iki GA-EAX süreci (run v26'nın birincil kolu)"),
    ("R3", EXACT_BETA_ARM, EXACT_ARM, "Birleşim + BetaTour kenarları / birleşim"),
    ("R4", EXACT_BETA_ARM, EXACT_NEAR_ARM, "Birleşim + BetaTour kenarları / birleşim + en kısa kenarlar"),
]
DESCRIPTIVE = [
    ("D1", EXACT_ARM, LKHMERGE_ARM, "Birleşimde tam çözüm / LKH'nin kendi tur birleştirmesi"),
    ("D2", EXACT_ARM, "lkh6", "Birleşimde tam çözüm / altı LKH süreci"),
    ("D3", EXACT_NEAR_ARM, EXACT_ARM, "Birleşim + en kısa kenarlar / birleşim"),
    ("D4", DUAL_ARM, LKHMERGE_ARM, "İki GA-EAX süreci / LKH'nin kendi tur birleştirmesi"),
    ("D5", BEST6_ARM, LKHMERGE_ARM, "Aynı altı tur: GA-EAX / LKH'nin kendi birleştirmesi"),
    ("D6", DUAL_ARM, "lkh6", "İki GA-EAX süreci / altı LKH süreci"),
]
COMPARISONS = READINGS + DESCRIPTIVE


def pct(a, b):
    return 100.0 * (a - b) / b


def load_finished():
    by_name = {x["name"]: x for x in INSTANCES}
    found = []
    for path in sorted((OUTPUT_ROOT / "instances").glob("*.json")):
        item = json.loads(path.read_text())
        if item.get("complete") and item["name"] in by_name:
            assert set(item["results"]) == set(ARM_NAMES) and item["xy_sha256"] == xy_hash(by_name[item["name"]])
            found.append(item)
    return sorted(found, key=lambda r: r["ordinal"])


records = load_finished()
rows = []
for item in records:
    exact = item["exact"]["arms"]
    row = dict(instance=item["name"], group=item["group"], family=item["family"], n=item["n"], budget_s=item["budget_s"],
               ordinal=item["ordinal"], order="-".join(item["order"]), cpu_model=item["hardware"]["cpu_model"],
               gpu=item["hardware"]["gpu"], session=item["session_id"][:8],
               best_at_cut=item["exact"]["best_at_cut"], union_edges=item["exact"]["union_edges"],
               k_extra=item["exact"]["k_extra"], pairwise_shared=item["exact"]["pairwise"]["pairwise_shared"])
    for arm in ARM_NAMES:
        row[arm] = item["results"][arm]
    for arm in TOUR_ARMS:
        row["outside_union6_" + arm] = item["edges_outside_union6"][arm]
    for arm, _ in EXACT_ARMS:
        a = exact[arm]
        row.update({f"{arm}_status": a["solver"]["status"], f"{arm}_optimal": bool(a["solver"]["optimal"]),
                    f"{arm}_solve_s": a["solver"]["solve_s"], f"{arm}_budget_s": a["exact_budget_s"],
                    f"{arm}_core": a["solver"]["core"], f"{arm}_free": a["solver"]["free"],
                    f"{arm}_bnb_nodes": a["solver"]["bnb_nodes"], f"{arm}_gap": a["solver"]["gap"],
                    f"{arm}_extra_used": a["extra_used"], f"{arm}_late": a["best_found_late"],
                    f"{arm}_after_deadline_s": a["finished_after_deadline_s"]})
    for arm, _ in EAX_ARMS:
        e = item["dual"]["arms"][arm]["eax"]
        row[arm + "_converged"] = None if e is None else bool(e["converged"])
    for key, first, second, _ in COMPARISONS:
        row[key] = pct(item["results"][first], item["results"][second])
    rows.append(row)
frame = pd.DataFrame(rows)
summary_dir = OUTPUT_ROOT / "summary"
frame.to_csv(summary_dir / "per_instance.csv", index=False)

groups = [("decision", DECISION_GROUPS)] + [("class:" + g, [g]) for g, *_ in SYNTHETIC] + \
         [("all", [g for g, *_ in SYNTHETIC])]
stats_rows = []
for label, members in groups:
    part = frame[frame["group"].isin(members)] if len(frame) else frame
    if not len(part):
        continue
    for key, first, second, text in COMPARISONS:
        s = B.paired_stats(part[key].to_numpy(), bootstrap_seed=27000 + len(stats_rows))
        stats_rows.append(dict(group=label, key=key, comparison=text, first=first, second=second, **s))
stats = pd.DataFrame(stats_rows)
stats.to_csv(summary_dir / "paired_stats.csv", index=False)

expected = sum(c for g, _, c, *_ in SYNTHETIC if g in DECISION_GROUPS)
decision_part = stats[stats["group"] == "decision"] if len(stats) else stats
complete = len(frame[frame["group"].isin(DECISION_GROUPS)]) == expected if len(frame) else False


def holds(key):
    row = decision_part[decision_part["key"] == key]
    if not len(row):
        return None
    r = row.iloc[0]
    return bool(r["win"] > r["loss"] and r["sign_p"] < 0.05)


readings = {key: holds(key) for key, *_ in READINGS}
exact_holds = bool(readings["R1"] or readings["R2"]) if complete else None
model_holds = bool(readings["R3"] and readings["R4"]) if complete else None
solver_rows = []
for arm, _ in EXACT_ARMS:
    for label, members in groups:
        part = frame[frame["group"].isin(members)] if len(frame) else frame
        if not len(part):
            continue
        solver_rows.append(dict(arm=arm, group=label, instances=len(part),
                                optimal=int(part[f"{arm}_optimal"].sum()), median_solve_s=float(part[f"{arm}_solve_s"].median()),
                                max_solve_s=float(part[f"{arm}_solve_s"].max()), median_core=float(part[f"{arm}_core"].median()),
                                extra_used_mean=float(part[f"{arm}_extra_used"].mean()),
                                late=int(part[f"{arm}_late"].sum())))
solver = pd.DataFrame(solver_rows)
solver.to_csv(summary_dir / "exact_solver.csv", index=False)
same_as_eax = int((frame[EXACT_ARM] == frame[BEST6_ARM]).sum()) if len(frame) else 0
main = dict(experiment=EXPERIMENT_NAME, instances=len(frame), decision_instances_expected=expected,
            decision_complete=bool(complete), readings=readings, exact_merging_holds=exact_holds,
            model_edges_holds=model_holds, exact_equal_to_eax_best6=same_as_eax,
            eax_best6_outside_union_mean=float(frame["outside_union6_" + BEST6_ARM].mean()) if len(frame) else None,
            eax_trials_outside_union_mean=float(frame["outside_union6_" + TRIALS_ARM].mean()) if len(frame) else None,
            test_mode=TEST_MODE, protocol=PROTOCOL, selfcheck=SELFCHECK,
            generated_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()))
write_json_atomic(summary_dir / "main.json", main)


def fmt(r):
    return (f"{int(r['win'])} / {int(r['tie'])} / {int(r['loss'])} | {r['mean_delta_pct']:+.4f} | "
            f"[{r['ci95_low_pct']:+.4f}, {r['ci95_high_pct']:+.4f}] | {r['sign_p']:.3g}")


lines = [f"# Run v27 (pilot): birleşim grafında tam çözüm — {EXPERIMENT_NAME}", "",
         f"Tamamlanan örnek: {len(frame)} / {len(INSTANCES)} (okumalar için {expected} örnek gerekir; tamam: {complete}).", "",
         "## Okumalar (10.000 düğüm, 12 örnek; ilk kol daha kısa / eşit / daha uzun | ortalama fark % | %95 aralık | işaret testi p)", ""]
for key, first, second, text in READINGS:
    row = decision_part[decision_part["key"] == key]
    lines.append(f"- {key} {text}: " + (fmt(row.iloc[0]) if len(row) else "veri yok") + f" — tutar mı: {readings[key]}")
lines += ["", f"Tam çözüm GA-EAX'ten kısa mı (R1 ya da R2): **{exact_holds}**. Model kenarları katkı veriyor mu (R3 ve R4): "
          f"**{model_holds}**.", "", "## Betimsel karşılaştırmalar (bütün örnekler)", ""]
for key, first, second, text in DESCRIPTIVE:
    row = stats[(stats["group"] == "all") & (stats["key"] == key)] if len(stats) else stats
    lines.append(f"- {key} {text}: " + (fmt(row.iloc[0]) if len(row) else "veri yok"))
lines += ["", "## Tam çözücü", ""]
for _, r in solver[solver["group"] == "all"].iterrows() if len(solver) else []:
    lines.append(f"- {r['arm']}: {r['optimal']}/{r['instances']} örnekte kanıtlanmış en kısa tur; çözüm süresi ortanca "
                 f"{r['median_solve_s']:.1f} s, en çok {r['max_solve_s']:.1f} s; çekirdek ortanca {r['median_core']:.0f} düğüm; "
                 f"kullanılan ek kenar ortalaması {r['extra_used_mean']:.2f}; süre sonrası bulunan sonuç: {r['late']}.")
lines += [f"- Tam çözüm (altı tur) ile GA-EAX'in (aynı altı tur) aynı uzunlukta olduğu örnek: {same_as_eax}.",
          f"- GA-EAX turlarının altı turun birleşiminde olmayan kenarları (ortalama): altı tur "
          f"{main['eax_best6_outside_union_mean']}, 102 tur {main['eax_trials_outside_union_mean']}.", ""]
(summary_dir / "report.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("\n".join(lines))

In [ ]:
import zipfile


def make_zip(name, include):
    """ZIP önce yerel diskte kurulur, sonra sonuç klasörüne (Drive) kopyalanır."""
    local = WORK_ROOT / name
    count = 0
    with zipfile.ZipFile(local, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as z:
        for path in sorted(OUTPUT_ROOT.rglob("*")):
            relative = path.relative_to(OUTPUT_ROOT)
            if not path.is_file() or path.suffix in {".zip", ".tmp"} or path.name.endswith(".tmp.npz"):
                continue
            if include(relative):
                z.write(path, str(relative))
                count += 1
    with zipfile.ZipFile(local) as z:
        assert z.testzip() is None
    digest = hashlib.sha256(local.read_bytes()).hexdigest()
    shutil.copy2(local, OUTPUT_ROOT / name)
    print(f"{name}: {count} dosya, {local.stat().st_size / 1e6:.1f} MB, SHA-256 {digest}")
    return local


# Sonuç ZIP'i: örnek kayıtları, her kolun son turu ve kesme anındaki altı tur (uzunlukların ve birleşimin bağımsız denetimi için),
# tablolar, rapor ve çalışan kod. Bunu gönderin.
result_zip = make_zip(EXPERIMENT_NAME + "_results.zip",
                      lambda rel: rel.parts[0] in {"instances", "tours", "summary", "code", "errors"}
                      or rel.name in {"manifest.json", "progress.json"})
if SAVE_TO_DRIVE:
    from google.colab import files
    files.download(str(result_zip))      # tarayıcıya iner
print("ZIP şu klasörde:", OUTPUT_ROOT)